# Face Swap v44 - RELIABLE BATCH IMAGE UPLOAD + STABLE VIDEO

**Swap ONE specific person's face across a whole video (not all faces).**

---



> **v44 buffalo_l download fix (2026-07-15):** detector-model downloads now resume after connection drops, retry automatically, validate the ZIP and required ONNX files, and extract safely before FaceAnalysis starts.

> **v44 upload fix (2026-07-15):** replaces the unreliable multi-file widget with a chunked browser uploader, shows current-file and overall upload percentages, and supports adding more target images without losing the existing ready list.

> **v43 batch-image feature (2026-07-15):** upload one source face and multiple target images, choose largest-face or all-faces replacement, use the same High Quality InsightFace + GFPGAN path as video, and export lossless PNG files in one ZIP.

> **v42 stability fix (2026-07-14):** the swap model now downloads from the official InsightFace release instead of a fragile third-party CDN redirect. Downloads use retries, a temporary `.part` file, exact-size validation, and SHA-256 verification before activation.

## What's new in v22

1. **Single-person targeting** - Pick exactly which person in the target video gets swapped, using a face index.
2. **Face preview & selection cell** - Auto-scans the first usable frame, draws boxes + numbers around every detected face, and lets you choose with a slider.
3. **Embedding-based tracking** - The chosen face is locked in by its InsightFace embedding, so the same person is followed across frames even if they move, turn or other faces enter/leave the shot.
4. **`process_video_optimized` extended** - Accepts `target_face_index` and `match_threshold` arguments; other detected faces are left untouched.
5. **Frame functions refactored** - `process_frame_enhanced` and `process_frame_optimized` now accept an optional target embedding and swap only the best-matching face.
6. **CUDA-stream path safely bypassed** when a target embedding is active (so we don't accidentally swap everyone).

All previous speed + quality features (Kalman, edge blend, color matching, temporal smoother, GFPGAN, TensorRT, batch, FP16) are preserved.

> Tip: if the chosen person briefly disappears from frame, the matcher will simply leave that frame untouched (no random face will be swapped).

---

## Preserved Features

### Speed
- TensorRT GPU acceleration
- CUDA Streams Pipeline (auto-bypassed when single-person mode is active)
- Batch Frame Processing
- Smart Frame Skipping
- FP16 Mixed Precision

### Quality
- v21 Speed-Optimized Kalman Tracking
- v21 Ultra Edge Blending (6-level Laplacian)
- v21 Advanced Color Matching (3-region)
- v17 Bidirectional Optical Flow
- v17 Expression Transfer
- v20 Temporal Smoother
- Interactive Parameter Controls

**Target Performance: 15-25 FPS**


In [ ]:
# ===================== CORE INSTALLATION (CUDA 12 FIXED + Python 3.13 SAFE) =====================
print('Installing Core Dependencies...')
print('='*70)

import subprocess
import sys
import shutil
import platform

PY_MAJOR, PY_MINOR = sys.version_info[:2]
print(f"Python: {platform.python_version()} (major={PY_MAJOR}, minor={PY_MINOR})")

# The original notebook pinned onnxruntime-gpu==1.20.1, which has NO wheel for
# Python 3.13 -> pip fails with a non-zero exit (the error you hit). Same for
# insightface 0.7.3, which only ships a source tarball and fails to compile its
# Cython "mesh_core_cython" extension on 3.13 + numpy 2 unless the build toolchain
# is installed first and the build happens with --no-build-isolation.
# This cell makes both installs Python-version-aware and resilient.

# --- CUDA version info (still useful for diagnostics) -------------------------
nvcc_path = shutil.which('nvcc')
if nvcc_path:
    result = subprocess.run([nvcc_path, '--version'], capture_output=True, text=True, check=False)
    print(f"CUDA Version Info:\n{result.stdout if result.stdout else result.stderr}")
else:
    print('CUDA Version Info: nvcc not found; continuing with runtime CUDA checks via PyTorch/ONNX.')

def pip_install(*packages, check=True, extra_args=()):
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', *extra_args, *packages]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"   [pip FAILED] {' '.join(packages)}")
        print("   " + (res.stderr.strip().splitlines() or ['unknown error'])[-1][:400])
        if check:
            raise RuntimeError(f"pip install failed for: {packages}")
    return res.returncode == 0

def pip_uninstall(*packages):
    cmd = [sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', *packages]
    subprocess.run(cmd, capture_output=True, text=True)

print('\n1. Core dependencies...')
pip_install('numpy>=1.24.0', 'opencv-python>=4.8.0', 'opencv-contrib-python>=4.8.0', check=False)

print('\n2. AI + CUDA packages (Colab CUDA 12 compatible)...')
# Remove any CPU or CUDA-13 onnxruntime first (CUDA-13 wheel needs libcudart.so.13).
pip_uninstall('onnxruntime', 'onnxruntime-gpu')

# Pick an onnxruntime-gpu that has a wheel for THIS Python AND bundles CUDA 12:
#   Python <= 3.12 -> 1.20.1 (stable, CUDA 12, matches original pin)
#   Python 3.13    -> 1.20.0 (first release with cp313 + CUDA 12 wheels)
ort_pkg = 'onnxruntime-gpu==1.20.1'
if (PY_MAJOR, PY_MINOR) >= (3, 13):
    ort_pkg = 'onnxruntime-gpu==1.20.0'
print(f"   Installing {ort_pkg} ...")
ok_ort = pip_install(ort_pkg, check=False)

# --- insightface: build toolchain + source build (Python 3.13 safe) -----------
# 0.7.3 has only a source tarball. On 3.13 the Cython/CMake build needs:
#   cython, cmake, ninja, scikit-build(+core), setuptools, wheel
# and must run with --no-build-isolation so it reuses the numpy already present
# (a fresh build env does not get the build deps correctly on 3.13).
print("   Pre-installing insightface build toolchain ...")
pip_install('cython', 'cmake', 'ninja', 'scikit-build-core', 'scikit-build',
            'setuptools', 'wheel', check=False)

print("   Pre-installing insightface runtime deps ...")
pip_install('onnx', 'requests', 'tqdm', 'scikit-image', 'scipy', 'matplotlib',
            'PyYAML', 'easydict', 'prettytable', 'albumentations', check=False)

print("   Installing insightface (source build, --no-build-isolation) ...")
ok_ifa = pip_install('insightface==0.7.3', extra_args=('--no-build-isolation',), check=False)

if not ok_ifa:
    print("   [insightface] primary build failed; retrying with a loose version pin ...")
    ok_ifa = pip_install('insightface', extra_args=('--no-build-isolation',), check=False)

if not ok_ifa:
    print("   [WARNING] insightface could not be built from source.")
    print("   If this runtime is Python 3.13, use a prebuilt insightface cp313 wheel:")
    print("     pip install https://github.com/Gourieff/Assets/raw/main/Insightface/insightface-0.7.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
    print("   (check the URL for your platform: Linux vs Windows, manylinux variants)")

# --- Import checks (do not hard-fail; print guidance instead) -----------------
import onnxruntime as ort
print(f"   ONNX Runtime: {ort.__version__}")
print(f"   Available providers: {ort.get_available_providers()}")
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    print('   WARNING: CUDAExecutionProvider is not visible to ONNX Runtime.')
    print('   Recheck Runtime > Change runtime type > T4 GPU, then restart runtime and run from the top.')

try:
    import torch
    print(f"   Using PyTorch: {torch.__version__}")
    print(f"   CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA version: {torch.version.cuda}")
except Exception as e:
    print(f"   PyTorch check skipped: {e}")

print('\n3. GFPGAN + Enhancement models...')
pip_install('basicsr', 'facexlib', 'realesrgan', 'gfpgan', check=False)

print('\n4. Video processing...')
pip_install('moviepy==1.0.3', 'imageio-ffmpeg', 'ffmpeg-python', check=False)

print('\n5. Additional dependencies...')
pip_install('tqdm', 'scikit-image', 'Pillow>=10.0.0', 'scipy', 'filterpy',
            'kornia', 'lpips', 'einops', 'ipywidgets', check=False)

print('\n' + '='*70)
print('Core packages installed!')
print('IMPORTANT: If Colab says packages were changed, restart runtime once, then run all cells from the top.')
print('='*70)


In [2]:
# ===================== CUDA STABLE MODE (TensorRT skipped intentionally) =====================
print('ðŸš€ CUDA Stable Mode')
print('='*70)

# TensorRT is optional. On current Colab images, pip may install TensorRT 11,
# while torch2trt still expects older TensorRT APIs such as CalibrationAlgoType.
# That warning does NOT mean CUDA is disabled. The fast face-swap path below uses
# InsightFace + onnxruntime-gpu with CUDAExecutionProvider.
TENSORRT_AVAILABLE = False
TORCH2TRT_AVAILABLE = False

print('\nTensorRT: skipped to avoid Colab TensorRT 11 / torch2trt mismatch')
print('CUDA speed path: kept ON via onnxruntime-gpu + CUDAExecutionProvider')
print('Quality: unaffected')
print('\n' + '='*70)
print('TensorRT Status: âšª Skipped')
print('CUDA Status: checked in the next GPU config cell')
print('='*70)


ðŸš€ Installing TensorRT (FIXED METHOD)...

1. Installing TensorRT...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1.2/1.2 MB 58.9 MB/s eta 0:00:00

2. Installing torch2trt from GitHub (FIXED)...
  Preparing metadata (setup.py) ... done
   âœ… TensorRT version: 11.0.0.114

âš ï¸ TensorRT/torch2trt installation issue: module 'tensorrt' has no attribute 'CalibrationAlgoType'
   Will use optimized standard inference as fallback
   (Still achieves 15-20 FPS with other optimizations)

TensorRT Status: âœ… Available

In [3]:
# ===================== GPU CHECK & OPTIMIZATION CONFIG =====================
import torch
import gc

print("ðŸ” Checking GPU and configuring optimizations...")
print("="*70)

if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"\nâœ… GPU ENABLED!")
    print(f"   Device: {gpu_name}")
    print(f"   Memory: {gpu_memory:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")

    # Enable optimizations
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    torch.cuda.empty_cache()
    gc.collect()

    # Configure batch size based on GPU memory
    if gpu_memory >= 14:  # T4 has ~15GB
        BATCH_SIZE = 8
        print(f"\n   High VRAM mode - Batch size: {BATCH_SIZE}")
    elif gpu_memory >= 8:
        BATCH_SIZE = 4
        print(f"\n   Medium VRAM mode - Batch size: {BATCH_SIZE}")
    else:
        BATCH_SIZE = 2
        print(f"\n   Low VRAM mode - Batch size: {BATCH_SIZE}")

    # Optimization flags - CUDA stays enabled; TensorRT is optional and skipped for Colab stability
    USE_TENSORRT = False
    USE_FP16 = True
    USE_CUDA_STREAMS = True
    USE_BATCH_PROCESSING = True
    USE_FRAME_SKIP = False  # Quality mode: never reuse old frames
    DET_SIZE = (640, 640)  # Better face boxes/landmarks than 320x320
    DEFAULT_OUTPUT_CRF = 14  # Lower = better quality/larger file

    print(f"\nâš¡ Optimization Config:")
    print(f"   TensorRT: {'âšª Skipped' if not USE_TENSORRT else 'âœ… Enabled'}")
    print(f"   FP16: {'âœ… Enabled' if USE_FP16 else 'âŒ Disabled'}")
    print(f"   CUDA Streams: {'âœ… Enabled' if USE_CUDA_STREAMS else 'âŒ Disabled'}")
    print(f"   Batch Processing: {'âœ… Enabled' if USE_BATCH_PROCESSING else 'âŒ Disabled'}")
    print(f"   Smart Frame Skip: {'âœ… Enabled' if USE_FRAME_SKIP else 'âŒ Disabled'}")
    print(f"   Detection Size: {DET_SIZE}")
    print(f"   Default Output CRF: {DEFAULT_OUTPUT_CRF}")

else:
    print("\nâŒ GPU NOT AVAILABLE!")
    print("Please enable GPU:")
    print("   1. Runtime > Change runtime type")
    print("   2. Select: T4 GPU")
    print("   3. Save and re-run")
    raise RuntimeError("GPU required for optimal performance")

print("\n" + "="*70)



ðŸ” Checking GPU and configuring optimizations...

âœ… GPU ENABLED!
   Device: Tesla T4
   Memory: 15.64 GB
   CUDA Version: 12.8

   High VRAM mode - Batch size: 8

âš¡ Optimization Config:
   TensorRT: âš ï¸ Fallback (still fast)
   FP16: âœ… Enabled
   CUDA Streams: âœ… Enabled
   Batch Processing: âœ… Enabled
   Smart Frame Skip: âœ… Enabled



In [4]:
# ===================== SPEED OPTIMIZATION MODULES (FIXED) =====================
import torch
import numpy as np
from typing import List, Callable, Optional, Tuple
import time
from collections import deque
import threading
from concurrent.futures import ThreadPoolExecutor

print("ðŸš€ Loading Speed Optimization Modules (FIXED)...")
print("="*70)

# ============ TensorRT Converter ===========
class TensorRTConverter:
    """Convert PyTorch models to TensorRT for 3-5x speedup."""

    def __init__(self, fp16=True, max_batch_size=8):
        self.fp16 = fp16
        self.max_batch_size = max_batch_size
        self.converted_models = {}
        self.enabled = USE_TENSORRT

    def convert(self, model, input_shape, model_name="model"):
        """Convert a PyTorch model to TensorRT."""
        if not self.enabled:
            print(f"   [TRT] Skipping {model_name} (TensorRT not available)")
            return model

        try:
            from torch2trt import torch2trt

            x = torch.randn(*input_shape).cuda()
            if self.fp16:
                x = x.half()
                model = model.half()

            model_trt = torch2trt(
                model,
                [x],
                fp16_mode=self.fp16,
                max_batch_size=self.max_batch_size
            )

            self.converted_models[model_name] = model_trt
            print(f"   âœ… {model_name} converted to TensorRT")
            return model_trt

        except Exception as e:
            print(f"   âš ï¸ {model_name} TensorRT conversion failed: {e}")
            return model

trt_converter = TensorRTConverter(fp16=USE_FP16, max_batch_size=BATCH_SIZE)
print("âœ… TensorRT Converter loaded")

# ============ CUDA Stream Pipeline (FIXED - NOW ACTUALLY USED) ===========
class CUDAStreamPipeline:
    """Parallel GPU processing using CUDA streams - NOW INTEGRATED."""

    def __init__(self, num_streams=3):
        self.enabled = USE_CUDA_STREAMS and torch.cuda.is_available()
        self.streams = []
        self.num_streams = num_streams

        if self.enabled:
            for i in range(num_streams):
                self.streams.append(torch.cuda.Stream())
            print(f"   âœ… {num_streams} CUDA streams created")

    def process_frame_parallel(self, frame, source_face, detect_fn, swap_fn):
        """Process single frame with parallel streams."""
        if not self.enabled:
            faces = detect_fn(frame)
            if len(faces) == 0:
                return frame, False
            result = frame.copy()
            for face in faces:
                result = swap_fn(result, source_face, face)
            return result, True

        # Use stream 0 for detection
        with torch.cuda.stream(self.streams[0]):
            faces = detect_fn(frame)

        torch.cuda.current_stream().wait_stream(self.streams[0])

        if len(faces) == 0:
            return frame, False

        # Use stream 1 for swapping
        with torch.cuda.stream(self.streams[1]):
            result = frame.copy()
            for face in faces:
                result = swap_fn(result, source_face, face)

        torch.cuda.current_stream().wait_stream(self.streams[1])
        return result, True

cuda_pipeline = CUDAStreamPipeline(num_streams=3)
print("âœ… CUDA Stream Pipeline loaded (FIXED)")

# ============ Batch Frame Processor (FIXED - NOW ACTUALLY USED) ===========
class BatchFrameProcessor:
    """Process multiple frames in batches - NOW INTEGRATED."""

    def __init__(self, batch_size=4):
        self.batch_size = batch_size
        self.enabled = USE_BATCH_PROCESSING
        self.frame_buffer = []
        self.result_buffer = []

    def add_frame(self, frame):
        """Add frame to buffer."""
        self.frame_buffer.append(frame)
        return len(self.frame_buffer) >= self.batch_size

    def process_batch(self, source_face, process_fn) -> List[Tuple[np.ndarray, bool]]:
        """Process all frames in buffer."""
        if not self.frame_buffer:
            return []

        results = []

        if self.enabled and len(self.frame_buffer) > 1:
            # Process in parallel using thread pool for I/O bound operations
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                for frame in self.frame_buffer:
                    result, success = process_fn(frame, source_face)
                    results.append((result, success))
        else:
            for frame in self.frame_buffer:
                result, success = process_fn(frame, source_face)
                results.append((result, success))

        self.frame_buffer = []
        return results

    def flush(self, source_face, process_fn) -> List[Tuple[np.ndarray, bool]]:
        """Process remaining frames in buffer."""
        return self.process_batch(source_face, process_fn)

batch_processor = BatchFrameProcessor(batch_size=BATCH_SIZE)
print(f"âœ… Batch Processor loaded (batch_size={BATCH_SIZE}) (FIXED)")

# ============ Frame Skip Cache ===========
class FrameSkipCache:
    """Cache for smart frame skipping when no face detected."""

    def __init__(self, max_skip=3):
        self.enabled = USE_FRAME_SKIP
        self.max_skip = max_skip
        self.skip_count = 0
        self.last_result = None
        self.last_had_face = False

    def should_skip(self) -> bool:
        """Check if we should skip processing."""
        if not self.enabled:
            return False
        return self.skip_count < self.max_skip and self.last_result is not None and not self.last_had_face

    def get_cached(self) -> np.ndarray:
        """Get cached result."""
        self.skip_count += 1
        return self.last_result

    def update(self, result: np.ndarray, had_face: bool):
        """Update cache with new result."""
        self.last_result = result
        self.last_had_face = had_face
        if had_face:
            self.skip_count = 0

frame_cache = FrameSkipCache(max_skip=3)
print("âœ… Frame Skip Cache loaded")

# ============ FPS Monitor ===========
class FPSMonitor:
    """Real-time FPS monitoring."""

    def __init__(self, window_size=30):
        self.frame_times = deque(maxlen=window_size)
        self.start_time = None
        self.total_frames = 0

    def start_frame(self):
        self.start_time = time.perf_counter()

    def end_frame(self):
        if self.start_time:
            elapsed = time.perf_counter() - self.start_time
            self.frame_times.append(elapsed)
            self.total_frames += 1

    def get_fps(self) -> float:
        if not self.frame_times:
            return 0.0
        avg_time = sum(self.frame_times) / len(self.frame_times)
        return 1.0 / avg_time if avg_time > 0 else 0.0

    def get_avg_frame_time_ms(self) -> float:
        if not self.frame_times:
            return 0.0
        return (sum(self.frame_times) / len(self.frame_times)) * 1000

fps_monitor = FPSMonitor(window_size=30)
print("âœ… FPS Monitor loaded")

print("\n" + "="*70)
print("ðŸš€ All Speed Optimization Modules Loaded (FIXED)!")
print("="*70)

ðŸš€ Loading Speed Optimization Modules (FIXED)...
âœ… TensorRT Converter loaded
   âœ… 3 CUDA streams created
âœ… CUDA Stream Pipeline loaded (FIXED)
âœ… Batch Processor loaded (batch_size=8) (FIXED)
âœ… Frame Skip Cache loaded
âœ… FPS Monitor loaded

ðŸš€ All Speed Optimization Modules Loaded (FIXED)!


In [5]:
# ===================== AI MODEL INITIALIZATION =====================
import os
import sys
import cv2
import numpy as np
import requests
import hashlib
import time
from pathlib import Path
from tqdm import tqdm
import gc
import torch

# Fix for torchvision compatibility
try:
    import torchvision.transforms.functional_tensor
except ModuleNotFoundError:
    import torchvision.transforms.functional as F
    sys.modules['torchvision.transforms.functional_tensor'] = F

print("Initializing AI Models...")
print("="*70)

# Validate ONNX Runtime before importing InsightFace. This catches the common
# Colab mismatch where a CUDA 13 onnxruntime-gpu wheel looks for libcudart.so.13.
try:
    import onnxruntime as ort
    print(f"ONNX Runtime: {ort.__version__}")
    print(f"ONNX providers: {ort.get_available_providers()}")
except ImportError as e:
    raise ImportError(
        "ONNX Runtime failed to import. Run the CORE INSTALLATION cell again. "
        "It picks an onnxruntime-gpu version whose wheel matches your Python and bundles CUDA 12 (removing CUDA 13 wheels). "
        f"Original error: {e}"
    )

try:
    import insightface
    from insightface.app import FaceAnalysis
except ImportError as e:
    raise ImportError(
        "InsightFace failed because ONNX Runtime is not usable. Restart the Colab runtime, "
        "run CORE INSTALLATION once, then run all cells from the top. "
        f"Original error: {e}"
    )


# Reliable buffalo_l model package setup.
# InsightFace's default downloader restarts after a dropped connection. This
# version preserves partial bytes, resumes with HTTP Range, retries, validates
# the ZIP, and only then activates the extracted model directory.
import zipfile
import shutil

INSIGHTFACE_HOME = Path.home() / '.insightface'
INSIGHTFACE_MODEL_ROOT = INSIGHTFACE_HOME / 'models'
BUFFALO_L_DIR = INSIGHTFACE_MODEL_ROOT / 'buffalo_l'
BUFFALO_L_ZIP = INSIGHTFACE_MODEL_ROOT / 'buffalo_l.zip'
BUFFALO_L_URL = 'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip'
BUFFALO_L_REQUIRED = {'det_10g.onnx', 'w600k_r50.onnx'}


def _buffalo_l_directory_ready(directory=BUFFALO_L_DIR):
    if not directory.is_dir():
        return False
    available = {
        path.name for path in directory.rglob('*.onnx')
        if path.is_file() and path.stat().st_size > 1024 * 1024
    }
    return BUFFALO_L_REQUIRED.issubset(available)


def _buffalo_l_zip_valid(zip_path):
    if not zip_path.is_file() or zip_path.stat().st_size < 1024 * 1024:
        return False
    try:
        with zipfile.ZipFile(zip_path, 'r') as archive:
            if archive.testzip() is not None:
                return False
            names = {Path(name).name for name in archive.namelist()}
            return BUFFALO_L_REQUIRED.issubset(names)
    except (zipfile.BadZipFile, OSError):
        return False


def _response_total_bytes(response, existing_bytes):
    content_range = response.headers.get('content-range', '')
    if '/' in content_range:
        try:
            return int(content_range.rsplit('/', 1)[1])
        except (TypeError, ValueError):
            pass
    content_length = response.headers.get('content-length')
    if content_length:
        try:
            length = int(content_length)
            return existing_bytes + length if response.status_code == 206 else length
        except (TypeError, ValueError):
            pass
    return None


def _download_buffalo_l_resumable(url=BUFFALO_L_URL, destination=BUFFALO_L_ZIP, attempts=8):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if _buffalo_l_zip_valid(destination):
        print(f'   Using verified cached buffalo_l package ({destination.stat().st_size / 1024**2:.1f} MB).')
        return

    last_error = None
    for attempt in range(1, attempts + 1):
        existing = destination.stat().st_size if destination.exists() else 0
        headers = {
            'User-Agent': 'Mozilla/5.0 Colab InsightFace resilient downloader',
            'Accept': 'application/octet-stream',
            'Accept-Encoding': 'identity',
            'Cache-Control': 'no-cache',
        }
        if existing:
            headers['Range'] = f'bytes={existing}-'
        try:
            action = f'resuming at {existing / 1024**2:.1f} MB' if existing else 'starting'
            print(f'   buffalo_l download attempt {attempt}/{attempts} ({action})...')
            with requests.get(
                url,
                stream=True,
                allow_redirects=True,
                headers=headers,
                timeout=(30, 300),
            ) as response:
                if response.status_code == 416:
                    if _buffalo_l_zip_valid(destination):
                        return
                    destination.unlink(missing_ok=True)
                    raise IOError('Server rejected the saved resume offset; restarting cleanly.')
                response.raise_for_status()

                resumed = existing > 0 and response.status_code == 206
                if existing > 0 and not resumed:
                    print('   Server did not accept resume; safely restarting this file.')
                    existing = 0
                mode = 'ab' if resumed else 'wb'
                expected_total = _response_total_bytes(response, existing)
                with open(destination, mode) as file_obj, tqdm(
                    total=expected_total,
                    initial=existing,
                    unit='B',
                    unit_scale=True,
                    desc='buffalo_l',
                ) as progress:
                    for chunk in response.iter_content(chunk_size=2 * 1024 * 1024):
                        if chunk:
                            file_obj.write(chunk)
                            progress.update(len(chunk))
                    file_obj.flush()
                    os.fsync(file_obj.fileno())

            final_size = destination.stat().st_size
            if expected_total is not None and final_size != expected_total:
                raise IOError(
                    f'Incomplete buffalo_l package: got {final_size:,} bytes; '
                    f'expected {expected_total:,}. The next retry will resume.'
                )
            if not _buffalo_l_zip_valid(destination):
                destination.unlink(missing_ok=True)
                raise IOError('Downloaded buffalo_l ZIP failed integrity/model validation; restarting.')
            print(f'   buffalo_l package verified ({final_size / 1024**2:.1f} MB).')
            return
        except Exception as error:
            last_error = error
            saved = destination.stat().st_size if destination.exists() else 0
            if attempt < attempts:
                wait_seconds = min(30, 3 * attempt)
                print(
                    f'   Download interrupted: {error}\n'
                    f'   Saved {saved / 1024**2:.1f} MB; retrying in {wait_seconds}s...'
                )
                time.sleep(wait_seconds)

    saved = destination.stat().st_size if destination.exists() else 0
    raise RuntimeError(
        'buffalo_l could not finish after 8 attempts. The partial download was kept '
        f'({saved / 1024**2:.1f} MB). Rerun this cell to continue. Last error: {last_error}'
    )


def _extract_buffalo_l_verified(zip_path=BUFFALO_L_ZIP, destination=BUFFALO_L_DIR):
    extracting = INSIGHTFACE_MODEL_ROOT / 'buffalo_l.extracting'
    shutil.rmtree(extracting, ignore_errors=True)
    extracting.mkdir(parents=True, exist_ok=True)
    extracting_root = extracting.resolve()

    try:
        with zipfile.ZipFile(zip_path, 'r') as archive:
            for member in archive.infolist():
                member_target = (extracting / member.filename).resolve()
                if os.path.commonpath([str(extracting_root), str(member_target)]) != str(extracting_root):
                    raise RuntimeError(f'Unsafe path in buffalo_l ZIP: {member.filename}')
            archive.extractall(extracting)

        nested = extracting / 'buffalo_l'
        extracted_model_dir = nested if nested.is_dir() else extracting
        if not _buffalo_l_directory_ready(extracted_model_dir):
            raise RuntimeError('Extracted buffalo_l folder is missing detector or recognition models.')

        shutil.rmtree(destination, ignore_errors=True)
        if extracted_model_dir == extracting:
            extracting.replace(destination)
        else:
            shutil.move(str(extracted_model_dir), str(destination))
            shutil.rmtree(extracting, ignore_errors=True)
        print(f'   buffalo_l models activated: {destination}')
    except Exception:
        shutil.rmtree(extracting, ignore_errors=True)
        raise


def _ensure_buffalo_l_models():
    INSIGHTFACE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    if _buffalo_l_directory_ready():
        print('   Using existing complete buffalo_l model directory.')
        return
    if BUFFALO_L_DIR.exists():
        print('   Removing incomplete buffalo_l model directory before verified extraction.')
        shutil.rmtree(BUFFALO_L_DIR, ignore_errors=True)
    _download_buffalo_l_resumable()
    _extract_buffalo_l_verified()
    if not _buffalo_l_directory_ready():
        raise RuntimeError('buffalo_l verification failed after extraction.')
    BUFFALO_L_ZIP.unlink(missing_ok=True)


print('\n0. Preparing InsightFace buffalo_l models with resume + retry...')
_ensure_buffalo_l_models()


cuda_ready = torch.cuda.is_available() and 'CUDAExecutionProvider' in ort.get_available_providers()
if cuda_ready:
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    provider_options = [
        {
            'device_id': 0,
            'arena_extend_strategy': 'kSameAsRequested',
            'gpu_mem_limit': 4 * 1024 * 1024 * 1024,
            'cudnn_conv_algo_search': 'EXHAUSTIVE',
            'cudnn_conv_use_max_workspace': '1',
            'do_copy_in_default_stream': True,
            'tunable_op_enable': '1',
            'tunable_op_tuning_enable': '1',
        },
        {},
    ]
    print('Using ONNX Runtime CUDAExecutionProvider')
else:
    providers = ['CPUExecutionProvider']
    provider_options = [{}]
    print('WARNING: ONNX Runtime CUDA provider is unavailable; using CPUExecutionProvider.')
    print('For speed, enable a Colab GPU, restart runtime, and run all cells from the top.')

# 1. Face Detection Model
print("\n1. Loading InsightFace detection model...")
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=providers,
    provider_options=provider_options
)
det_size = globals().get("DET_SIZE", (640, 640))
face_app.prepare(ctx_id=0 if cuda_ready else -1, det_size=det_size)
print(f"   Face detection ready ({det_size[0]}x{det_size[1]})")

# 2. Face Swap Model
print("\n2. Loading inswapper_128 model...")
model_dir = Path.home() / '.insightface' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'inswapper_128.onnx'

# Official InsightFace v0.7 release asset. Avoid third-party Hugging Face CDN
# redirects: their short-lived signed Xet URLs can intermittently return HTTP 403.
model_url = 'https://github.com/deepinsight/insightface/releases/download/v0.7/inswapper_128.onnx'
model_size = 554253681
model_sha256 = 'e4a3f08c753cb72d04e10aa0f7dbe3deebbf39567d4ead6dce08e98aa49e16af'

def _sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file_obj:
        for chunk in iter(lambda: file_obj.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _model_is_valid(path):
    if not path.is_file() or path.stat().st_size != model_size:
        return False
    print('   Verifying cached model checksum...')
    return _sha256(path) == model_sha256

def _download_verified_model(url, destination, attempts=3):
    part_path = destination.with_suffix(destination.suffix + '.part')
    headers = {
        'User-Agent': 'Mozilla/5.0 Colab InsightFace model installer',
        'Accept': 'application/octet-stream',
        'Cache-Control': 'no-cache',
    }
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            part_path.unlink(missing_ok=True)
            print(f'   Downloading official model (~554MB), attempt {attempt}/{attempts}...')
            with requests.get(
                url, stream=True, allow_redirects=True, headers=headers, timeout=(30, 300)
            ) as response:
                response.raise_for_status()
                total_size = int(response.headers.get('content-length') or model_size)
                with open(part_path, 'wb') as file_obj, tqdm(
                    total=total_size, unit='B', unit_scale=True, desc='Downloading'
                ) as pbar:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file_obj.write(chunk)
                            pbar.update(len(chunk))
            if part_path.stat().st_size != model_size:
                raise IOError(
                    f'Incomplete model: got {part_path.stat().st_size:,} bytes; expected {model_size:,}'
                )
            print('   Verifying downloaded model checksum...')
            actual_sha256 = _sha256(part_path)
            if actual_sha256 != model_sha256:
                raise IOError(f'Model checksum mismatch: {actual_sha256}')
            part_path.replace(destination)
            return
        except Exception as error:
            last_error = error
            part_path.unlink(missing_ok=True)
            if attempt < attempts:
                wait_seconds = 3 * attempt
                print(f'   Download failed ({error}); retrying in {wait_seconds}s...')
                time.sleep(wait_seconds)
    raise RuntimeError(
        'Could not download the official InsightFace inswapper model after 3 attempts. '
        'Check Colab internet access, then rerun this cell. '
        f'Last error: {last_error}'
    )

if not _model_is_valid(model_path):
    if model_path.exists():
        print('   Cached model is incomplete or corrupt; replacing it safely.')
        model_path.unlink()
    _download_verified_model(model_url, model_path)
else:
    print('   Using verified cached inswapper model.')

swapper = insightface.model_zoo.get_model(
    str(model_path),
    providers=providers,
    provider_options=provider_options
)
print("   Face swapper ready")

# 3. GFPGAN Face Restoration (optional for quality mode)
print("\n3. Loading GFPGAN v1.4 (optional enhancement)...")
GFPGAN_AVAILABLE = False
gfpgan_restorer = None

try:
    from gfpgan import GFPGANer

    gfpgan_model_dir = Path('/content/gfpgan_models')
    gfpgan_model_dir.mkdir(exist_ok=True)
    gfpgan_model_path = gfpgan_model_dir / 'GFPGANv1.4.pth'

    if not gfpgan_model_path.exists():
        print("   Downloading GFPGAN v1.4...")
        gfpgan_url = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
        response = requests.get(gfpgan_url, stream=True, timeout=180)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        with open(gfpgan_model_path, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, desc='Downloading GFPGAN') as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

    gfpgan_restorer = GFPGANer(
        model_path=str(gfpgan_model_path),
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    GFPGAN_AVAILABLE = True
    print("   GFPGAN ready")
except Exception as e:
    print(f"   GFPGAN not available: {e}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("\n" + "="*70)
print("All AI Models Initialized!")
print(f"   ONNX provider mode: {'CUDA' if cuda_ready else 'CPU'}")
print(f"   GFPGAN: {'Available' if GFPGAN_AVAILABLE else 'Not loaded'}")
print("="*70)


ðŸ¤– Initializing AI Models...

1. Loading InsightFace detection model...
download_path: /root/.insightface/models/buffalo_l


100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 281857/281857 [00:03<00:00, 87558.79KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '4294967296', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'

Downloading: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 554M/554M [00:03<00:00, 159MB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '4294967296', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
inswapper-shape: [1, 3, 128, 128]
   âœ… Face swapper ready

3. Loading GFPGAN v1.4 (optional enhancement)...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /content/gfpgan/weights/detection_Resnet50_Final.pth



100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 104M/104M [00:00<00:00, 429MB/s] 


Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth" to /content/gfpgan/weights/parsing_parsenet.pth



100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 81.4M/81.4M [00:00<00:00, 439MB/s]


   âœ… GFPGAN ready

âœ… All AI Models Initialized!
   GFPGAN: âœ… Available


In [6]:
# ===================== v21 ENHANCED MOTION TRACKING (Speed-Optimized) =====================
from filterpy.kalman import KalmanFilter
import numpy as np
import cv2

class SpeedOptimizedKalmanTracker:
    """v21: Speed-optimized Kalman filter for 15-25 FPS target.

    Features:
    - 4-state model (position + velocity) - faster than 6-state
    - Adaptive noise based on motion magnitude
    - Efficient numpy operations
    - Configurable buffer size
    """
    def __init__(self, num_landmarks=5, process_noise=0.005, buffer_size=50):
        self.buffer_size = buffer_size
        self.base_process_noise = process_noise
        self.trackers = []
        self.history = []
        self.prev_landmarks = None

        for _ in range(num_landmarks):
            kf = KalmanFilter(dim_x=4, dim_z=2)
            kf.F = np.array([[1, 0, 1, 0],
                             [0, 1, 0, 1],
                             [0, 0, 1, 0],
                             [0, 0, 0, 1]])
            kf.H = np.array([[1, 0, 0, 0],
                             [0, 1, 0, 0]])
            kf.Q *= process_noise
            kf.R *= 0.05
            kf.P *= 10
            self.trackers.append(kf)
        self.initialized = False

    def estimate_motion(self, landmarks):
        if self.prev_landmarks is None:
            return 0.0
        return np.linalg.norm(np.array(landmarks) - np.array(self.prev_landmarks))

    def adapt_process_noise(self, motion):
        scale = 1.0 + min(motion / 50.0, 5.0)
        for kf in self.trackers:
            kf.Q = np.eye(4) * (self.base_process_noise * scale)

    def update(self, landmarks):
        landmarks = np.array(landmarks).reshape(-1, 2)

        if not self.initialized:
            for i, lm in enumerate(landmarks):
                self.trackers[i].x[:2] = lm
            self.initialized = True
            self.prev_landmarks = landmarks.copy()
            return landmarks

        motion = self.estimate_motion(landmarks)
        self.adapt_process_noise(motion)

        smoothed = []
        for i, lm in enumerate(landmarks):
            self.trackers[i].predict()
            self.trackers[i].update(lm)
            smoothed.append(self.trackers[i].x[:2].flatten())

        result = np.array(smoothed)
        self.history.append(result.copy())
        if len(self.history) > self.buffer_size:
            self.history.pop(0)

        self.prev_landmarks = landmarks.copy()
        return result

    def reset(self):
        self.initialized = False
        self.history = []
        self.prev_landmarks = None

# Initialize global tracker
kalman_tracker = SpeedOptimizedKalmanTracker(num_landmarks=5, buffer_size=50)
print('[v21] Speed-Optimized Kalman Tracker loaded')
print('      4-state model for maximum performance')
print('      Adaptive noise based on motion')


[v21] Speed-Optimized Kalman Tracker loaded
      4-state model for maximum performance
      Adaptive noise based on motion


In [7]:
# ===================== v21 ULTRA EDGE BLENDING (Speed-Optimized) =====================
import cv2
import numpy as np
from scipy.ndimage import distance_transform_edt

class FastEdgeBlender:
    """v21: Speed-optimized edge blending for 15-25 FPS.

    Features:
    - Zone-specific feathering (hairline/jawline)
    - 6-level Laplacian pyramid (optimized from 8)
    - Optional Poisson blending (can disable for speed)
    """
    def __init__(self, hairline_feather=45, jawline_feather=35, use_poisson=False):
        self.hairline_feather = hairline_feather
        self.jawline_feather = jawline_feather
        self.use_poisson = use_poisson

    def create_gradient_mask(self, mask, landmarks=None):
        h, w = mask.shape[:2]
        _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        dist = distance_transform_edt(binary_mask)
        dist = cv2.normalize(dist, None, 0, 1, cv2.NORM_MINMAX).astype(np.float32)

        if landmarks is not None and len(landmarks) >= 5:
            eye_y = int((landmarks[0][1] + landmarks[1][1]) / 2)
            mouth_y = int((landmarks[3][1] + landmarks[4][1]) / 2)

            hairline_mask = np.zeros_like(binary_mask, dtype=np.float32)
            hairline_mask[:eye_y - int(0.2*(mouth_y - eye_y)), :] = 1.0

            jawline_mask = np.zeros_like(binary_mask, dtype=np.float32)
            jawline_mask[mouth_y:, :] = 1.0

            middle_mask = np.clip(1.0 - hairline_mask - jawline_mask, 0, 1)
        else:
            hairline_mask = np.zeros((h, w), dtype=np.float32)
            hairline_mask[:h//3, :] = 1.0
            jawline_mask = np.zeros((h, w), dtype=np.float32)
            jawline_mask[2*h//3:, :] = 1.0
            middle_mask = 1.0 - hairline_mask - jawline_mask

        hairline_blurred = cv2.GaussianBlur(binary_mask.astype(np.float32),
            (self.hairline_feather*2+1, self.hairline_feather*2+1), 0)
        jawline_blurred = cv2.GaussianBlur(binary_mask.astype(np.float32),
            (self.jawline_feather*2+1, self.jawline_feather*2+1), 0)
        middle_blurred = cv2.GaussianBlur(binary_mask.astype(np.float32),
            (40*2+1, 40*2+1), 0)

        combined = (hairline_blurred * hairline_mask +
                   jawline_blurred * jawline_mask +
                   middle_blurred * middle_mask)

        final_mask = combined * 0.6 + dist * 0.4
        final_mask = cv2.normalize(final_mask, None, 0, 255, cv2.NORM_MINMAX)
        return final_mask.astype(np.uint8)

    def alpha_blend(self, source, target, mask):
        mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR).astype(np.float32) / 255.0
        source_f = source.astype(np.float32)
        target_f = target.astype(np.float32)
        blended = source_f * mask_3ch + target_f * (1 - mask_3ch)
        return np.clip(blended, 0, 255).astype(np.uint8)

def gaussian_pyramid(img, levels=6):
    G = img.copy().astype(np.float64)
    gpA = [G]
    for i in range(levels):
        G = cv2.pyrDown(G)
        gpA.append(G)
    return gpA

def laplacian_pyramid(img, levels=6):
    gp = gaussian_pyramid(img, levels)
    lp = []
    for i in range(levels):
        size = (gp[i].shape[1], gp[i].shape[0])
        ge = cv2.pyrUp(gp[i+1], dstsize=size)
        L = cv2.subtract(gp[i], ge)
        lp.append(L)
    lp.append(gp[levels])
    return lp

def blend_laplacian_pyramids(lpA, lpB, gpMask):
    LS = []
    for la, lb, gm in zip(lpA, lpB, gpMask):
        if len(gm.shape) == 2:
            gm = cv2.cvtColor(gm.astype(np.uint8), cv2.COLOR_GRAY2BGR).astype(np.float64)
        gm = gm / 255.0
        ls = la * gm + lb * (1.0 - gm)
        LS.append(ls)
    return LS

def reconstruct_from_laplacian_pyramid(lp):
    img = lp[-1]
    for i in range(len(lp)-2, -1, -1):
        size = (lp[i].shape[1], lp[i].shape[0])
        img = cv2.pyrUp(img, dstsize=size)
        img = cv2.add(img, lp[i])
    return img

def fast_laplacian_blend(face, target, mask, levels=6, feather_radius=40, landmarks=None):
    """v21: Speed-optimized multi-scale blending."""
    blender = FastEdgeBlender(hairline_feather=feather_radius+10, jawline_feather=feather_radius)
    soft_mask = blender.create_gradient_mask(mask, landmarks)

    lpFace = laplacian_pyramid(face.astype(np.float64), levels)
    lpTarget = laplacian_pyramid(target.astype(np.float64), levels)
    gpMask = gaussian_pyramid(soft_mask.astype(np.float64), levels)

    lpBlend = blend_laplacian_pyramids(lpFace, lpTarget, gpMask)
    result = reconstruct_from_laplacian_pyramid(lpBlend)
    return np.clip(result, 0, 255).astype(np.uint8)

# Initialize global blender
edge_blender = FastEdgeBlender(hairline_feather=45, jawline_feather=35)
print('[v21] Speed-Optimized Edge Blending loaded')
print('      6-level Laplacian pyramid')
print('      Zone-specific feathering')


[v21] Speed-Optimized Edge Blending loaded
      6-level Laplacian pyramid
      Zone-specific feathering


In [8]:
# ===================== v21 ADVANCED COLOR MATCHING (Speed-Optimized) =====================
import cv2
import numpy as np

class FastColorMatcher:
    """v21: Speed-optimized multi-region color matching.

    Features:
    - YCrCb skin detection
    - 3-region matching (forehead, cheeks, chin) - optimized from 5
    - Luminance preservation
    - Temporal smoothing
    """
    def __init__(self, strength=0.8):
        self.strength = strength
        self.prev_stats = None
        self.temporal_alpha = 0.7

    def detect_skin_mask(self, image):
        ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
        lower = np.array([0, 133, 77], dtype=np.uint8)
        upper = np.array([255, 173, 127], dtype=np.uint8)
        skin_mask = cv2.inRange(ycrcb, lower, upper)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel)
        return skin_mask

    def get_region_stats(self, image, mask):
        if mask.sum() == 0:
            return None
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB).astype(np.float32)
        mask_bool = mask > 0
        stats = {}
        for i, channel in enumerate(['L', 'A', 'B']):
            values = lab[mask_bool, i]
            if len(values) > 0:
                stats[channel] = {'mean': np.mean(values), 'std': np.std(values) + 1e-6}
        return stats

    def match_region(self, source_img, target_img, mask, preserve_lum=0.3):
        if mask.sum() == 0:
            return source_img

        src_stats = self.get_region_stats(source_img, mask)
        tgt_stats = self.get_region_stats(target_img, mask)

        if src_stats is None or tgt_stats is None:
            return source_img

        src_lab = cv2.cvtColor(source_img, cv2.COLOR_BGR2LAB).astype(np.float32)
        original_L = src_lab[:, :, 0].copy()
        mask_float = mask.astype(np.float32) / 255.0

        for i, channel in enumerate(['L', 'A', 'B']):
            if channel in src_stats and channel in tgt_stats:
                normalized = (src_lab[:, :, i] - src_stats[channel]['mean']) / src_stats[channel]['std']
                matched = normalized * tgt_stats[channel]['std'] + tgt_stats[channel]['mean']
                blended = src_lab[:, :, i] * (1 - self.strength * mask_float) + matched * (self.strength * mask_float)
                src_lab[:, :, i] = blended

        if preserve_lum > 0:
            src_lab[:, :, 0] = src_lab[:, :, 0] * (1 - preserve_lum) + original_L * preserve_lum

        src_lab = np.clip(src_lab, 0, 255).astype(np.uint8)
        return cv2.cvtColor(src_lab, cv2.COLOR_LAB2BGR)

    def match_fast(self, swapped_face, target_face, landmarks=None):
        """Fast 3-region color matching."""
        h, w = swapped_face.shape[:2]
        result = swapped_face.copy()
        skin_mask = self.detect_skin_mask(target_face)

        if landmarks is None or len(landmarks) < 5:
            return self.match_region(result, target_face, skin_mask)

        eye_y = int((landmarks[0][1] + landmarks[1][1]) / 2)
        mouth_y = int((landmarks[3][1] + landmarks[4][1]) / 2)

        # Forehead region
        forehead_mask = np.zeros((h, w), dtype=np.uint8)
        forehead_mask[:eye_y - 5, :] = 255
        forehead_combined = cv2.bitwise_and(forehead_mask, skin_mask)
        if forehead_combined.sum() > 100:
            result = self.match_region(result, target_face, forehead_combined, 0.4)

        # Cheeks region (combined left + right)
        cheeks_mask = np.zeros((h, w), dtype=np.uint8)
        cheeks_mask[eye_y:mouth_y, :] = 255
        cheeks_combined = cv2.bitwise_and(cheeks_mask, skin_mask)
        if cheeks_combined.sum() > 100:
            result = self.match_region(result, target_face, cheeks_combined, 0.25)

        # Chin region
        chin_mask = np.zeros((h, w), dtype=np.uint8)
        chin_mask[mouth_y:, :] = 255
        chin_combined = cv2.bitwise_and(chin_mask, skin_mask)
        if chin_combined.sum() > 100:
            result = self.match_region(result, target_face, chin_combined, 0.25)

        return result

# Initialize global color matcher
color_matcher = FastColorMatcher(strength=0.8)
print('[v21] Speed-Optimized Color Matching loaded')
print('      YCrCb skin detection')
print('      3-region matching with luminance preservation')


[v21] Speed-Optimized Color Matching loaded
      YCrCb skin detection
      3-region matching with luminance preservation


In [9]:
# ===================== v17 BIDIRECTIONAL OPTICAL FLOW =====================
import cv2
import numpy as np

class BidirectionalOpticalFlow:
    """v17: Bidirectional optical flow with consistency checking.

    Features:
    - Forward-backward consistency check
    - Outlier rejection
    - Confidence-weighted blending
    """
    def __init__(self, threshold=1.0):
        self.threshold = threshold
        self.prev_gray = None
        self.prev_flow = None

    def compute_flow(self, frame1_gray, frame2_gray):
        return cv2.calcOpticalFlowFarneback(
            frame1_gray, frame2_gray, None,
            pyr_scale=0.5, levels=4, winsize=13,
            iterations=3, poly_n=5, poly_sigma=1.2,
            flags=cv2.OPTFLOW_FARNEBACK_GAUSSIAN
        )

    def check_consistency(self, forward_flow, backward_flow):
        h, w = forward_flow.shape[:2]
        y, x = np.mgrid[0:h, 0:w].astype(np.float32)

        warped_x = np.clip(x + forward_flow[:, :, 0], 0, w-1).astype(np.float32)
        warped_y = np.clip(y + forward_flow[:, :, 1], 0, h-1).astype(np.float32)

        backward_at_warped = cv2.remap(backward_flow, warped_x, warped_y, cv2.INTER_LINEAR)

        consistency_error = np.sqrt(
            (forward_flow[:, :, 0] + backward_at_warped[:, :, 0])**2 +
            (forward_flow[:, :, 1] + backward_at_warped[:, :, 1])**2
        )

        confidence = np.exp(-consistency_error / self.threshold)
        return confidence

    def process_frame(self, current_frame):
        current_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)

        if self.prev_gray is None:
            self.prev_gray = current_gray
            return current_frame, np.ones_like(current_gray, dtype=np.float32)

        forward_flow = self.compute_flow(self.prev_gray, current_gray)
        backward_flow = self.compute_flow(current_gray, self.prev_gray)
        confidence = self.check_consistency(forward_flow, backward_flow)

        self.prev_gray = current_gray
        self.prev_flow = forward_flow

        return current_frame, confidence

    def reset(self):
        self.prev_gray = None
        self.prev_flow = None

# Initialize optical flow processor
optical_flow = BidirectionalOpticalFlow(threshold=1.0)
print('[v17] Bidirectional Optical Flow loaded')
print('      Forward-backward consistency checking')


[v17] Bidirectional Optical Flow loaded
      Forward-backward consistency checking


In [10]:
# ===================== v17 EXPRESSION TRANSFER =====================
import cv2
import numpy as np

class ExpressionTransfer:
    """v17: Expression transfer with intensity control.

    Features:
    - Facial landmark interpolation
    - Expression intensity control (0-100%)
    - Smooth blending
    """
    def __init__(self, intensity=0.7):
        self.intensity = intensity
        self.prev_source_landmarks = None
        self.prev_target_landmarks = None

    def interpolate_landmarks(self, source_lm, target_lm, factor=None):
        if factor is None:
            factor = self.intensity
        source_lm = np.array(source_lm)
        target_lm = np.array(target_lm)
        return source_lm * (1 - factor) + target_lm * factor

    def compute_expression_delta(self, neutral_lm, expressive_lm):
        neutral_lm = np.array(neutral_lm)
        expressive_lm = np.array(expressive_lm)
        face_size = np.maximum(np.max(neutral_lm, axis=0) - np.min(neutral_lm, axis=0), 1)
        return (expressive_lm - neutral_lm) / face_size

    def apply_expression_delta(self, landmarks, delta, intensity=None):
        if intensity is None:
            intensity = self.intensity
        landmarks = np.array(landmarks)
        face_size = np.max(landmarks, axis=0) - np.min(landmarks, axis=0)
        return landmarks + delta * face_size * intensity

    def blend_expressions(self, source_lm, target_lm):
        blended_lm = self.interpolate_landmarks(source_lm, target_lm)
        self.prev_source_landmarks = source_lm
        self.prev_target_landmarks = target_lm
        return blended_lm

# Initialize expression transfer
expression_transfer = ExpressionTransfer(intensity=0.7)
print('[v17] Expression Transfer loaded')
print('      Landmark interpolation with intensity control')


[v17] Expression Transfer loaded
      Landmark interpolation with intensity control


In [11]:
# ===================== v20 BIDIRECTIONAL TEMPORAL SMOOTHER =====================
import numpy as np
import cv2
from collections import deque

class TemporalSmoother:
    """v20: Lightweight bidirectional temporal smoothing.

    Features:
    - Gaussian-weighted temporal blending
    - Scene change detection
    - Adaptive window based on motion
    """
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.landmark_buffer = deque(maxlen=window_size)
        self.weights = self._gaussian_weights(window_size)
        self.scene_threshold = 30.0

    def _gaussian_weights(self, size):
        x = np.arange(size) - size // 2
        weights = np.exp(-x**2 / (2 * (size/4)**2))
        return weights / weights.sum()

    def detect_scene_change(self, frame1, frame2):
        gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        diff = np.mean(np.abs(gray1.astype(float) - gray2.astype(float)))
        return diff > self.scene_threshold

    def add_landmarks(self, landmarks):
        self.landmark_buffer.append(np.array(landmarks))

    def smooth_landmarks(self):
        if len(self.landmark_buffer) < 3:
            return self.landmark_buffer[-1] if self.landmark_buffer else None

        landmarks = np.array(list(self.landmark_buffer))
        n = len(landmarks)
        weights = self._gaussian_weights(n)
        smoothed = np.sum(landmarks * weights[:, None, None], axis=0)
        return smoothed

# Initialize temporal smoother
temporal_smoother = TemporalSmoother(window_size=5)
print('[v20] Temporal Smoother loaded')
print('      Gaussian-weighted blending')
print('      Scene change detection')


[v20] Temporal Smoother loaded
      Gaussian-weighted blending
      Scene change detection


In [12]:
# ===================== INTERACTIVE PARAMETER CONTROLS =====================
from ipywidgets import IntSlider, FloatSlider, Dropdown, Checkbox
import ipywidgets as widgets
from IPython.display import display

TEMPORAL_BUFFER_DEFAULT = 50

print('Interactive Parameter Controls')
print('='*70)

# Temporal Buffer
temporal_buffer_slider = IntSlider(
    value=TEMPORAL_BUFFER_DEFAULT, min=20, max=60, step=5,
    description='Temporal Buffer:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# Feather Radius
feather_radius_slider = IntSlider(
    value=40, min=20, max=60, step=5,
    description='Feather Radius:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# Expression Intensity
expression_intensity_slider = FloatSlider(
    value=0.7, min=0.0, max=1.0, step=0.1,
    description='Expression Intensity:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# Color Match Strength
color_match_strength_slider = FloatSlider(
    value=0.8, min=0.0, max=1.0, step=0.1,
    description='Color Match:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# GFPGAN Weight
gfpgan_weight_slider = FloatSlider(
    value=0.85, min=0.5, max=1.0, step=0.05,
    description='GFPGAN Weight:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# Quality Preset
quality_preset = Dropdown(
    options=['SPEED (15-25 FPS)', 'BALANCED (10-15 FPS)', 'QUALITY (5-10 FPS)'],
    value='SPEED (15-25 FPS)',
    description='Quality Preset:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='300px')
)

# Enable Enhanced Features
enable_enhanced = Checkbox(
    value=True,
    description='Enable v21 Enhanced Features',
    style={'description_width': 'initial'}
)

print('\nTEMPORAL STABILITY:')
display(temporal_buffer_slider)

print('\nEDGE BLENDING:')
display(feather_radius_slider)

print('\nEXPRESSION & COLOR:')
display(expression_intensity_slider)
display(color_match_strength_slider)

print('\nRESTORATION:')
display(gfpgan_weight_slider)
display(quality_preset)
display(enable_enhanced)

print('\n' + '='*70)
print('Parameters will be applied when processing starts.')


Interactive Parameter Controls

TEMPORAL STABILITY:


IntSlider(value=50, description='Temporal Buffer:', layout=Layout(width='500px'), max=60, min=20, step=5, stylâ€¦


EDGE BLENDING:


IntSlider(value=40, description='Feather Radius:', layout=Layout(width='500px'), max=60, min=20, step=5, styleâ€¦


EXPRESSION & COLOR:


FloatSlider(value=0.7, description='Expression Intensity:', layout=Layout(width='500px'), max=1.0, style=Slideâ€¦

... (2 outputs left out)

FloatSlider(value=0.85, description='GFPGAN Weight:', layout=Layout(width='500px'), max=1.0, min=0.5, step=0.0â€¦

Dropdown(description='Quality Preset:', layout=Layout(width='300px'), options=('SPEED (15-25 FPS)', 'BALANCED â€¦

Checkbox(value=True, description='Enable v21 Enhanced Features', style=DescriptionStyle(description_width='iniâ€¦


Parameters will be applied when processing starts.


In [ ]:
# ===================== ENHANCED PROCESS FRAME FUNCTION (v22 single-person) =====================
import numpy as np
import cv2


def _face_embedding(face):
    emb = getattr(face, "normed_embedding", None)
    if emb is None:
        emb = getattr(face, "embedding", None)
    return emb


def _cosine_sim(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    na = np.linalg.norm(a) + 1e-8
    nb_ = np.linalg.norm(b) + 1e-8
    return float(np.dot(a, b) / (na * nb_))


def _face_area(face):
    bbox = getattr(face, "bbox", None)
    if bbox is None:
        return 0.0
    x1, y1, x2, y2 = np.asarray(bbox, dtype=np.float32)
    return max(0.0, float(x2 - x1)) * max(0.0, float(y2 - y1))


def select_largest_face(faces):
    if not faces:
        return None
    return max(faces, key=_face_area)


def match_target_face(faces, target_embedding, threshold=0.35):
    """Return (face, score) for the best embedding match, or (None, score)."""
    if not faces or target_embedding is None:
        return None, -1.0
    best, best_score = None, -1.0
    for f in faces:
        emb = _face_embedding(f)
        if emb is None:
            continue
        s = _cosine_sim(emb, target_embedding)
        if s > best_score:
            best_score, best = s, f
    if best is None or best_score < threshold:
        return None, best_score
    return best, best_score


def choose_target_face_with_fallback(faces, target_embedding=None, threshold=0.25, force_fallback=True):
    if not faces:
        return None, 'none', -1.0
    if target_embedding is not None:
        matched, score = match_target_face(faces, target_embedding, threshold)
        if matched is not None:
            return matched, 'embedding', score
        if not force_fallback:
            return None, 'miss', score
        return select_largest_face(faces), 'largest_fallback', score
    return select_largest_face(faces), 'largest', -1.0


def process_frame_enhanced(frame, source_face, target_faces, use_enhancement=True,
                           use_color_match=True, use_edge_blend=True,
                           target_embedding=None, match_threshold=0.35):
    """v22: Enhanced frame processing with single-person mode.

    If target_embedding is provided, only the face matching that embedding will
    be swapped. Otherwise behaves like v21 (swap all detected faces).
    """
    if target_faces is None or len(target_faces) == 0:
        return frame

    # Single-person mode: filter to the matching face only
    if target_embedding is not None:
        matched, match_mode, match_score = choose_target_face_with_fallback(
            target_faces, target_embedding, match_threshold, force_fallback=False)
        if matched is None:
            return frame
        target_faces = [matched]

    result = frame.copy()

    for target_face in target_faces:
        try:
            landmarks = None
            if hasattr(target_face, "kps") and target_face.kps is not None:
                landmarks = target_face.kps

            swapped = swapper.get(result, target_face, source_face, paste_back=True)

            if use_enhancement:
                if landmarks is not None:
                    smoothed_landmarks = kalman_tracker.update(landmarks)
                    temporal_smoother.add_landmarks(smoothed_landmarks)

                if use_color_match and landmarks is not None:
                    bbox = target_face.bbox.astype(int)
                    x1, y1, x2, y2 = (max(0, bbox[0]), max(0, bbox[1]),
                                      min(result.shape[1], bbox[2]),
                                      min(result.shape[0], bbox[3]))
                    face_region = swapped[y1:y2, x1:x2]
                    original_region = frame[y1:y2, x1:x2]
                    if face_region.size > 0 and original_region.size > 0:
                        local_landmarks = landmarks - np.array([x1, y1])
                        matched_region = color_matcher.match_fast(
                            face_region, original_region, local_landmarks)
                        swapped[y1:y2, x1:x2] = matched_region

            result = swapped
        except Exception as e:
            print(f"Frame processing error: {e}")
            continue

    if use_enhancement and "gfpgan_restorer" in globals() and gfpgan_restorer is not None:
        try:
            weight = globals().get("gfpgan_weight_slider", None)
            weight = float(weight.value) if weight is not None else 0.75
            _, _, result = gfpgan_restorer.enhance(
                result, has_aligned=False, only_center_face=False,
                paste_back=True, weight=weight)
        except Exception:
            pass

    return result


print("[v22] Enhanced Process Frame loaded (single-person aware)")
print("      Pass target_embedding to swap only one specific person")


In [ ]:
# ===================== OPTIMIZED FACE SWAP FUNCTIONS (v25 force-real-swap) =====================
import cv2
import numpy as np
from typing import Optional, List, Tuple

print("Loading Optimized Face Swap Functions (v25 force-real-swap)...")
print("=" * 70)

FORCE_LARGEST_FACE_SWAP = True
MIN_SWAP_CHANGE_RATIO = 0.0004  # Full-frame guard; face ROI check is stricter and more useful


def detect_faces_fast(frame: np.ndarray) -> list:
    return face_app.get(frame)


def frame_change_ratio(before: np.ndarray, after: np.ndarray) -> float:
    if before is None or after is None or before.shape != after.shape:
        return 0.0
    diff = cv2.absdiff(before, after)
    return float(np.mean(diff) / 255.0)


def face_roi_change_ratio(before: np.ndarray, after: np.ndarray, face, expand: float = 0.35) -> float:
    if before is None or after is None or before.shape != after.shape or face is None:
        return 0.0
    bbox = getattr(face, "bbox", None)
    if bbox is None:
        return frame_change_ratio(before, after)
    h, w = before.shape[:2]
    x1, y1, x2, y2 = np.asarray(bbox, dtype=np.float32)
    bw, bh = x2 - x1, y2 - y1
    x1 = int(max(0, x1 - bw * expand))
    y1 = int(max(0, y1 - bh * expand))
    x2 = int(min(w, x2 + bw * expand))
    y2 = int(min(h, y2 + bh * expand))
    if x2 <= x1 or y2 <= y1:
        return frame_change_ratio(before, after)
    return frame_change_ratio(before[y1:y2, x1:x2], after[y1:y2, x1:x2])


def swap_face_fast(frame: np.ndarray, source_face, target_face) -> np.ndarray:
    return swapper.get(frame, target_face, source_face, paste_back=True)


def enhance_face_fast(frame: np.ndarray, use_gfpgan: bool = False, weight: float = 0.75) -> np.ndarray:
    if not use_gfpgan or not GFPGAN_AVAILABLE or gfpgan_restorer is None:
        return frame
    try:
        _, _, output = gfpgan_restorer.enhance(
            frame, has_aligned=False, only_center_face=False, paste_back=True, weight=weight
        )
        return output
    except Exception as e:
        print(f"GFPGAN enhance skipped: {e}")
        return frame


def choose_faces_for_swap(faces, target_embedding=None, match_threshold: float = 0.20, force_largest: bool = True):
    if not faces:
        return [], "none", -1.0
    if force_largest:
        return [select_largest_face(faces)], "largest_forced", -1.0
    if target_embedding is not None:
        matched, mode, score = choose_target_face_with_fallback(
            faces, target_embedding, match_threshold, force_fallback=False
        )
        return ([matched] if matched is not None else []), mode, score
    return [select_largest_face(faces)], "largest", -1.0


def process_frame_optimized(
    frame: np.ndarray,
    source_face,
    use_enhancement: bool = False,
    target_embedding=None,
    match_threshold: float = 0.20,
    force_largest: bool = True,
) -> Tuple[np.ndarray, bool]:
    """v25: process one frame and force a real target face selection by default."""
    try:
        faces = detect_faces_fast(frame)
        selected_faces, mode, score = choose_faces_for_swap(
            faces,
            target_embedding=target_embedding,
            match_threshold=match_threshold,
            force_largest=force_largest,
        )
        if not selected_faces:
            return frame, False

        result = frame.copy()
        for target_face in selected_faces:
            result = swap_face_fast(result, source_face, target_face)

        if use_enhancement and frame_change_ratio(frame, result) >= MIN_SWAP_CHANGE_RATIO:
            result = enhance_face_fast(result, use_gfpgan=True, weight=0.70)

        roi_change = max(face_roi_change_ratio(frame, result, f) for f in selected_faces)
        full_change = frame_change_ratio(frame, result)
        return result, (roi_change >= 0.003 or full_change >= MIN_SWAP_CHANGE_RATIO)
    except Exception as e:
        print(f"process_frame_optimized skipped frame: {e}")
        return frame, False


print("Optimized face swap functions loaded (v25 force-real-swap)")
print("=" * 70)


In [ ]:
# ===================== VIDEO PROCESSING ENGINE (v23 high-quality single-person) =====================
import cv2
import numpy as np
from tqdm import tqdm
import time
import os
import subprocess
from pathlib import Path
from moviepy.editor import VideoFileClip, AudioFileClip
import gc

print("Loading Video Processing Engine (v23 HQ)...")
print("=" * 70)


def _replace_ext(path: str, suffix: str, ext: str) -> str:
    base, _ = os.path.splitext(path)
    return f"{base}{suffix}{ext}"


def _open_high_quality_writer(output_path: str, fps: float, width: int, height: int):
    """Prefer a lossless FFV1/MKV intermediate to avoid double lossy MP4 encoding."""
    writer_attempts = [
        (_replace_ext(output_path, "_temp_lossless", ".mkv"), "FFV1", "lossless FFV1/MKV"),
        (_replace_ext(output_path, "_temp_hq", ".avi"), "MJPG", "high-quality MJPEG/AVI"),
        (_replace_ext(output_path, "_temp", ".mp4"), "avc1", "H.264/MP4 fallback"),
        (_replace_ext(output_path, "_temp", ".mp4"), "mp4v", "MPEG-4/MP4 fallback"),
    ]
    for temp_path, fourcc_name, label in writer_attempts:
        fourcc = cv2.VideoWriter_fourcc(*fourcc_name)
        writer = cv2.VideoWriter(temp_path, fourcc, fps, (width, height))
        if writer.isOpened():
            print(f"   Intermediate: {label}")
            print(f"   Temp file: {temp_path}")
            return writer, temp_path, label
        writer.release()
    raise RuntimeError("Could not open any video writer for the output video")


def _mux_high_quality_video(temp_output: str, target_video_path: str, output_path: str,
                            crf: int = 14, preset: str = "veryfast") -> bool:
    """Encode final MP4 once; prefer NVENC when available, fallback to libx264."""
    if os.path.exists(output_path):
        os.remove(output_path)

    nvenc_cmd = [
        "ffmpeg", "-y",
        "-i", temp_output,
        "-i", target_video_path,
        "-map", "0:v:0",
        "-map", "1:a?",
        "-c:v", "h264_nvenc",
        "-preset", "p6",
        "-tune", "hq",
        "-rc", "vbr",
        "-cq", str(crf),
        "-b:v", "0",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        output_path,
    ]
    x264_cmd = [
        "ffmpeg", "-y",
        "-i", temp_output,
        "-i", target_video_path,
        "-map", "0:v:0",
        "-map", "1:a?",
        "-c:v", "libx264",
        "-preset", preset,
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        output_path,
    ]
    try:
        for label, cmd in [("NVENC", nvenc_cmd), ("libx264", x264_cmd)]:
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0 and os.path.exists(output_path):
                print(f"   Final MP4 encoded with {label}")
                return True
            print(f"   {label} mux failed; stderr tail:")
            print("\n".join(result.stderr.splitlines()[-5:]))
        return False
    except Exception as e:
        print(f"   ffmpeg HQ mux failed: {e}")
        return False


def process_video_optimized(
    source_image_path: str,
    target_video_path: str,
    output_path: str = "/content/output_swapped.mp4",
    use_enhancement: bool = True,
    max_frames: int = None,
    show_progress: bool = True,
    target_face_index: int = None,
    match_threshold: float = 0.20,
    target_embedding=None,
    force_largest: bool = True,
    output_crf: int = None,
    output_preset: str = "medium",
) -> dict:
    """v23: Fast single-person swap with higher-quality final MP4.

    Quality upgrade over v22:
    - Uses a lossless/high-quality temporary video when supported by Colab ffmpeg.
    - Avoids MoviePy's second fixed-bitrate re-encode.
    - Final MP4 is encoded once with libx264 CRF 16 + veryfast preset.

    Lower CRF = better quality/larger file. Recommended range: 14-18.
    """

    single_person = target_face_index is not None or target_embedding is not None

    print("\nStarting Video Processing (v23 HQ)...")
    print(f"   Source image: {source_image_path}")
    print(f"   Target video: {target_video_path}")
    print(f"   Mode: {'FORCE LARGEST FACE' if force_largest else ('SINGLE PERSON' if single_person else 'ALL FACES')}")
    if output_crf is None:
        output_crf = globals().get("DEFAULT_OUTPUT_CRF", 14)
    print(f"   Enhancement: {'ON' if use_enhancement else 'OFF (max speed)'}")
    print(f"   Output quality: CRF {output_crf}, preset {output_preset}")

    # --- Load source face ---
    print("\n1. Loading source face...")
    source_img = cv2.imread(source_image_path)
    if source_img is None:
        raise ValueError(f"Could not load source image: {source_image_path}")
    source_faces = detect_faces_fast(source_img)
    if len(source_faces) == 0:
        raise ValueError("No face detected in source image")
    source_face = select_largest_face(source_faces)
    print(f"   Source face detected; selected largest of {len(source_faces)} face(s)")

    # --- Open video ---
    print("\n2. Opening target video...")
    cap = cv2.VideoCapture(target_video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {target_video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if max_frames:
        total_frames = min(total_frames, max_frames)

    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps}")
    print(f"   Frames to process: {total_frames}")

    # --- Resolve target_face_index -> embedding (if needed) ---
    if target_embedding is None and target_face_index is not None:
        print(f"\n2b. Locking on target_face_index={target_face_index} ...")
        chosen = None
        scanned = 0
        while scanned < 60:
            ret, frame = cap.read()
            if not ret:
                break
            faces = detect_faces_fast(frame)
            if faces and target_face_index < len(faces):
                faces_sorted = sorted(faces, key=lambda f: float(f.bbox[0]))
                chosen = faces_sorted[target_face_index]
                break
            scanned += 1
        cap.release()
        cap = cv2.VideoCapture(target_video_path)  # rewind

        if chosen is None:
            raise ValueError(
                f"Could not find face index {target_face_index} in the first 60 frames.")
        target_embedding = _face_embedding(chosen)
        if target_embedding is None:
            raise ValueError("Selected face has no embedding")
        print("   Target person locked.")

    # --- Setup writer ---
    print("\n3. Setting up high-quality output...")
    out, temp_output, intermediate_label = _open_high_quality_writer(output_path, fps, width, height)

    # --- Process frames ---
    print("\n4. Processing frames...")
    fps_monitor.frame_times.clear()
    fps_monitor.total_frames = 0
    frame_cache.last_result = None
    frame_cache.skip_count = 0

    processed_count = 0
    success_count = 0
    unchanged_count = 0
    frame_times = []
    pbar = tqdm(total=total_frames, desc="Processing", disable=not show_progress)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    try:
        frame_buffer = []
        while processed_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            start_time = time.perf_counter()

            if USE_BATCH_PROCESSING:
                frame_buffer.append(frame)
                if (len(frame_buffer) >= BATCH_SIZE
                        or processed_count + len(frame_buffer) >= total_frames):
                    with torch.cuda.amp.autocast(enabled=USE_FP16):
                        for buf_frame in frame_buffer:
                            result, success = process_frame_optimized(
                                buf_frame, source_face,
                                use_enhancement=use_enhancement,
                                target_embedding=target_embedding,
                                match_threshold=match_threshold,
                                force_largest=force_largest,
                            )
                            frame_cache.update(result, success)
                            out.write(result)
                            elapsed = time.perf_counter() - start_time
                            frame_times.append(elapsed)
                            fps_monitor.frame_times.append(elapsed)
                            fps_monitor.total_frames += 1
                            processed_count += 1
                            if success:
                                success_count += 1
                            else:
                                unchanged_count += 1
                            current_fps = fps_monitor.get_fps()
                            pbar.set_postfix({"FPS": f"{current_fps:.1f}",
                                              "OK": f"{success_count}"})
                            pbar.update(1)
                            start_time = time.perf_counter()
                    frame_buffer = []
                    if processed_count % (BATCH_SIZE * 5) == 0:
                        torch.cuda.empty_cache() if torch.cuda.is_available() else None
            else:
                result, success = process_frame_optimized(
                    frame, source_face,
                    use_enhancement=use_enhancement,
                    target_embedding=target_embedding,
                    match_threshold=match_threshold,
                    force_largest=force_largest,
                )
                elapsed = time.perf_counter() - start_time
                frame_times.append(elapsed)
                fps_monitor.frame_times.append(elapsed)
                fps_monitor.total_frames += 1
                out.write(result)
                processed_count += 1
                if success:
                    success_count += 1
                else:
                    unchanged_count += 1
                current_fps = fps_monitor.get_fps()
                pbar.set_postfix({"FPS": f"{current_fps:.1f}",
                                  "OK": f"{success_count}"})
                pbar.update(1)
                if processed_count % 50 == 0:
                    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    finally:
        pbar.close()
        cap.release()
        out.release()

    # --- Audio merge / final high-quality encode ---
    print("\n5. Creating final high-quality MP4...")
    mux_ok = _mux_high_quality_video(
        temp_output=temp_output,
        target_video_path=target_video_path,
        output_path=output_path,
        crf=output_crf,
        preset=output_preset,
    )
    if mux_ok:
        print("   Final MP4 created with high-quality ffmpeg settings")
        try:
            os.remove(temp_output)
        except Exception:
            pass
    else:
        print("   Falling back to previous MoviePy audio merge")
        try:
            video_clip = VideoFileClip(temp_output)
            audio_clip = VideoFileClip(target_video_path).audio
            if audio_clip is not None:
                final_clip = video_clip.set_audio(audio_clip)
                final_clip.write_videofile(
                    output_path, codec="libx264", audio_codec="aac",
                    bitrate="16000k", preset=output_preset,
                    verbose=False, logger=None,
                )
                final_clip.close(); video_clip.close()
                os.remove(temp_output)
            else:
                os.rename(temp_output, output_path)
            print("   Audio merged")
        except Exception as e:
            print(f"   Audio merge failed: {e}")
            if os.path.exists(temp_output):
                os.rename(temp_output, output_path)

    avg_fps = 1.0 / np.mean(frame_times) if frame_times else 0
    min_fps = 1.0 / np.max(frame_times) if frame_times else 0
    max_fps = 1.0 / np.min(frame_times) if frame_times else 0

    stats = {
        "output_path": output_path,
        "total_frames": processed_count,
        "changed_frames": success_count,
        "unchanged_frames": unchanged_count,
        "avg_fps": avg_fps, "min_fps": min_fps, "max_fps": max_fps,
        "avg_frame_time_ms": float(np.mean(frame_times) * 1000) if frame_times else 0,
        "total_time_seconds": float(sum(frame_times)) if frame_times else 0,
        "enhancement_enabled": use_enhancement,
        "mode": "single_person" if single_person else "all_faces",
        "intermediate": intermediate_label,
        "output_crf": output_crf,
        "output_preset": output_preset,
    }

    print("\n" + "=" * 70)
    print("PROCESSING COMPLETE")
    print("=" * 70)
    print(f"   Output: {output_path}")
    print(f"   Frames processed: {processed_count}")
    print(f"   Frames with target swapped: {success_count} "
          f"({100*success_count/max(processed_count,1):.1f}%)")
    print(f"   Average face-swap FPS: {avg_fps:.1f}")
    if avg_fps >= 15:
        print(f"   TARGET ACHIEVED ({avg_fps:.1f} FPS >= 15 FPS)")
    print("=" * 70)
    return stats


print("Video Processing Engine loaded (v23 HQ)")
print("=" * 70)


In [ ]:
# ===================== FUNCTION HEALTH CHECK =====================
print("Function Health Check")
print("=" * 70)

required_globals = [
    'face_app', 'swapper', 'providers',
    'detect_faces_fast', 'swap_face_fast', 'enhance_face_fast', 'frame_change_ratio', 'face_roi_change_ratio',
    'process_frame_enhanced', 'process_frame_optimized', 'process_video_optimized',
    'match_target_face', 'select_largest_face', '_face_embedding', '_cosine_sim',
]

missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects/functions: {missing}")

import onnxruntime as ort
print(f"ONNX Runtime version: {ort.__version__}")
print(f"ONNX providers available: {ort.get_available_providers()}")
print(f"Notebook provider order: {providers}")
print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Tiny CPU-only utility checks that do not require uploaded files.
assert abs(_cosine_sim(np.array([1, 0], dtype=np.float32), np.array([1, 0], dtype=np.float32)) - 1.0) < 1e-5
assert _face_embedding(type('DummyFace', (), {'normed_embedding': np.array([1, 0], dtype=np.float32)})()) is not None

print("All core functions are defined and basic utility checks passed.")
print("Run FILE UPLOAD, TARGET PERSON SELECTION, then PROCESS VIDEO for the full GPU/video test.")
print("=" * 70)


In [ ]:
# ===================== UPLOAD UI MOVED =====================
print('Upload is now handled by the styled UI in the final processing cell.')
print('Run cells up to STABLE FAST FORCE SWAP MODE, then use the upload panels and Swap Faces button.')

In [ ]:
# ===================== TARGET PERSON SELECTION DISABLED FOR UI MODE =====================
selected_target_embedding = None
selected_face_index = 0
print('Target selection is handled by the v30 stable UI workflow. Current mode: force largest face for stability.')

In [ ]:
# ===================== QUICK PREVIEW MOVED INTO UI WORKFLOW =====================
print('Preview checks are handled by the UI workflow before full processing starts.')

In [ ]:
# ===================== PROCESS VIDEO (v30 STABLE FAST FORCE SWAP) =====================
# Target: keep the stable every-frame neural swap look and remove cache/tracking overhead.
# Quality note: N=1 is the quality/stability target. N>1 is optional and may reduce stability.
import cv2
import os
import time
import gc
import numpy as np
import torch
try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm
from queue import Queue
from threading import Thread

print("STABLE FAST FORCE SWAP MODE (v41 REFERENCE-STABLE SPATIAL LOCK)")
print("=" * 70)

# Selectable UI modes. Fast mode keeps speed; slower work only runs when selected.
FAST_QUALITY_PROFILE = "TURBO_15_25_FPS"

_FAST_PROFILES = {
    "TURBO_15_25_FPS": {
        "label": "Stable Turbo Motion",
        "det_size": (320, 320),
        "swap_every_n": 3,
        "detect_every_n": 1,
        "crf": 18,
        "preset": "ultrafast",
        "smart": True,
        "motion_threshold": 0.022,
        "pose_motion_threshold": 0.045,
        "gfpgan": False,
        "match_threshold": 0.10,
    },
    "FAST_FACE_ONLY": {
        "label": "Fast Face Only",
        "det_size": (384, 384),
        "swap_every_n": 4,
        "detect_every_n": 4,
        "crf": 16,
        "preset": "ultrafast",
        "smart": False,
        "motion_threshold": 0.030,
        "pose_motion_threshold": 0.070,
        "gfpgan": False,
        "match_threshold": 0.11,
    },
    "BETTER_FACE_COVERAGE": {
        "label": "Better Face Coverage",
        "det_size": (512, 512),
        "swap_every_n": 2,
        "detect_every_n": 2,
        "crf": 11,
        "preset": "veryfast",
        "smart": False,
        "motion_threshold": 0.022,
        "pose_motion_threshold": 0.055,
        "gfpgan": False,
        "match_threshold": 0.14,
    },
    "HIGH_QUALITY": {
        "label": "High Quality",
        "det_size": (768, 768),
        "swap_every_n": 1,
        "detect_every_n": 1,
        "crf": 8,
        "preset": "medium",
        "smart": False,
        "motion_threshold": 0.018,
        "pose_motion_threshold": 0.050,
        "gfpgan": True,
        "match_threshold": 0.16,
    },
}
_profile = _FAST_PROFILES.get(FAST_QUALITY_PROFILE, _FAST_PROFILES["TURBO_15_25_FPS"])

FAST_DET_SIZE = _profile["det_size"]
FAST_SWAP_EVERY_N = _profile["swap_every_n"]
FAST_OUTPUT_CRF = _profile["crf"]
FAST_OUTPUT_PRESET = _profile["preset"]
FAST_SMART_NEURAL = _profile["smart"]
FAST_MOTION_NEURAL_THRESHOLD = _profile["motion_threshold"]
FAST_USE_GFPGAN_ENHANCEMENT = _profile["gfpgan"]
FAST_MIN_OUTPUT_FPS = 24.0  # Guard against bad/very-low source FPS metadata like 4-7 fps.
FAST_TARGET_OUTPUT_FPS = 30.0
FAST_PROGRESS_UPDATE_EVERY = 3
FAST_BBOX_ALPHA_NEURAL = 0.82
FAST_BBOX_ALPHA_CACHE = 0.70
FAST_BBOX_MAX_CENTER_STEP = 0.28
FAST_BBOX_MAX_SCALE_STEP = 0.12

print(f"   Profile: {FAST_QUALITY_PROFILE}")
print(f"   Detector: {FAST_DET_SIZE}")
print(f"   Enhancement: {'ON' if FAST_USE_GFPGAN_ENHANCEMENT else 'OFF'}")
print(f"   Neural swap cadence: every {FAST_SWAP_EVERY_N} frame(s)")
print(f"   Motion-triggered fresh swaps: {'ON' if FAST_SMART_NEURAL else 'OFF'}")
print(f"   Output quality: CRF/CQ {FAST_OUTPUT_CRF}, {FAST_OUTPUT_PRESET} preset")
print(f"   FPS guard: normalize below {FAST_MIN_OUTPUT_FPS:g} fps to {FAST_TARGET_OUTPUT_FPS:g} fps")
print("=" * 70)

try:
    face_app.prepare(ctx_id=0 if torch.cuda.is_available() else -1, det_size=FAST_DET_SIZE)
    print(f"   Re-prepared detector: {FAST_DET_SIZE}")
except Exception as e:
    print(f"   Detector re-prepare skipped: {e}")


def _clip_bbox(bbox, width, height, pad_ratio_x=0.20, pad_ratio_y=0.25):
    x1, y1, x2, y2 = [float(v) for v in bbox]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    pad_x = max(10.0, bw * pad_ratio_x)
    pad_y = max(10.0, bh * pad_ratio_y)
    x1 = int(max(0, x1 - pad_x))
    y1 = int(max(0, y1 - pad_y))
    x2 = int(min(width, x2 + pad_x))
    y2 = int(min(height, y2 + pad_y))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def _target_face_bbox(frame, target_embedding=None, target_match_threshold=0.20):
    faces = detect_faces_fast(frame)
    if not faces:
        return None
    if target_embedding is not None:
        face, _, _ = choose_target_face_with_fallback(
            faces, target_embedding, target_match_threshold, force_fallback=False
        )
    else:
        face = select_largest_face(faces)
    if face is None or getattr(face, "bbox", None) is None:
        return None
    return np.asarray(face.bbox, dtype=np.float32)


def _largest_face_bbox(frame):
    return _target_face_bbox(frame)


def _face_roi_motion_score(previous_frame, current_frame, bbox):
    if previous_frame is None or current_frame is None or bbox is None:
        return 1.0
    h, w = current_frame.shape[:2]
    roi = _clip_bbox(bbox, w, h, pad_ratio_x=0.10, pad_ratio_y=0.12)
    if roi is None:
        return 1.0
    x1, y1, x2, y2 = roi
    prev = previous_frame[y1:y2, x1:x2]
    curr = current_frame[y1:y2, x1:x2]
    if prev.size == 0 or curr.size == 0 or prev.shape != curr.shape:
        return 1.0
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
    return float(np.mean(cv2.absdiff(prev_gray, curr_gray)) / 255.0)


def _smooth_bbox(previous_bbox, current_bbox, alpha=0.35,
                 max_center_step=FAST_BBOX_MAX_CENTER_STEP,
                 max_scale_step=FAST_BBOX_MAX_SCALE_STEP):
    if previous_bbox is None:
        return None if current_bbox is None else np.asarray(current_bbox, dtype=np.float32)
    if current_bbox is None:
        return np.asarray(previous_bbox, dtype=np.float32)
    previous_bbox = np.asarray(previous_bbox, dtype=np.float32)
    current_bbox = np.asarray(current_bbox, dtype=np.float32)

    prev_center = (previous_bbox[:2] + previous_bbox[2:]) * 0.5
    curr_center = (current_bbox[:2] + current_bbox[2:]) * 0.5
    prev_size = np.maximum(previous_bbox[2:] - previous_bbox[:2], 1.0)
    curr_size = np.maximum(current_bbox[2:] - current_bbox[:2], 1.0)

    max_step_px = max(4.0, float(np.linalg.norm(prev_size)) * max_center_step)
    center_delta = curr_center - prev_center
    center_dist = float(np.linalg.norm(center_delta))
    if center_dist > max_step_px:
        curr_center = prev_center + center_delta * (max_step_px / (center_dist + 1e-6))

    scale_ratio = np.clip(curr_size / prev_size, 1.0 - max_scale_step, 1.0 + max_scale_step)
    curr_size = prev_size * scale_ratio

    smooth_center = prev_center * (1.0 - alpha) + curr_center * alpha
    smooth_size = prev_size * (1.0 - alpha) + curr_size * alpha
    half = smooth_size * 0.5
    return np.concatenate([smooth_center - half, smooth_center + half]).astype(np.float32)


def _track_bbox_optical_flow(previous_frame, current_frame, previous_bbox):
    """Track the locked face box cheaply between detector frames."""
    if previous_frame is None or current_frame is None or previous_bbox is None:
        return None
    h, w = current_frame.shape[:2]
    roi = _clip_bbox(previous_bbox, w, h, pad_ratio_x=0.18, pad_ratio_y=0.20)
    if roi is None:
        return None
    x1, y1, x2, y2 = roi
    prev_gray = cv2.cvtColor(previous_frame, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)
    mask = np.zeros_like(prev_gray)
    mask[y1:y2, x1:x2] = 255
    pts = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=48,
        qualityLevel=0.01,
        minDistance=5,
        blockSize=5,
        mask=mask,
    )
    if pts is None or len(pts) < 6:
        return None
    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray,
        curr_gray,
        pts,
        None,
        winSize=(21, 21),
        maxLevel=2,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 12, 0.03),
    )
    if next_pts is None or status is None:
        return None
    good_prev = pts[status.reshape(-1) == 1].reshape(-1, 2)
    good_next = next_pts[status.reshape(-1) == 1].reshape(-1, 2)
    if len(good_prev) < 6:
        return None
    deltas = good_next - good_prev
    dx, dy = np.median(deltas, axis=0)
    if not np.isfinite(dx) or not np.isfinite(dy):
        return None
    prev_bbox = np.asarray(previous_bbox, dtype=np.float32)
    bw = max(1.0, prev_bbox[2] - prev_bbox[0])
    bh = max(1.0, prev_bbox[3] - prev_bbox[1])
    max_shift = max(8.0, 0.22 * max(bw, bh))
    dx = float(np.clip(dx, -max_shift, max_shift))
    dy = float(np.clip(dy, -max_shift, max_shift))
    tracked = prev_bbox + np.array([dx, dy, dx, dy], dtype=np.float32)
    tracked[0::2] = np.clip(tracked[0::2], 0, w - 1)
    tracked[1::2] = np.clip(tracked[1::2], 0, h - 1)
    if tracked[2] <= tracked[0] or tracked[3] <= tracked[1]:
        return None
    return tracked.astype(np.float32)


def _bbox_iou(a, b):
    if a is None or b is None:
        return 0.0
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    return float(inter / (area_a + area_b - inter + 1e-6))


def _bbox_center_distance_ratio(a, b):
    if a is None or b is None:
        return 999.0
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    ca = (a[:2] + a[2:]) * 0.5
    cb = (b[:2] + b[2:]) * 0.5
    size = np.maximum(a[2:] - a[:2], 1.0)
    return float(np.linalg.norm(ca - cb) / (np.linalg.norm(size) + 1e-6))


def _choose_spatial_locked_face(faces, target_embedding, match_threshold, lock_bbox=None):
    if not faces:
        return None, 'none', -1.0
    matched, score = match_target_face(faces, target_embedding, match_threshold) if target_embedding is not None else (None, -1.0)
    if matched is not None:
        return matched, 'embedding', score
    if lock_bbox is None:
        return None, 'miss', score
    best_face, best_rank = None, (-1.0, -999.0)
    for face in faces:
        bbox = getattr(face, 'bbox', None)
        if bbox is None:
            continue
        iou = _bbox_iou(lock_bbox, bbox)
        dist = _bbox_center_distance_ratio(lock_bbox, bbox)
        # Spatial fallback is only allowed near the already locked track.
        if iou >= 0.18 or dist <= 0.38:
            rank = (iou, -dist)
            if rank > best_rank:
                best_rank = rank
                best_face = face
    if best_face is not None:
        return best_face, 'spatial_lock', score
    return None, 'miss', score


def _process_frame_turbo_locked(frame, source_face, target_embedding=None, match_threshold=0.20, lock_bbox=None, use_enhancement=False):
    try:
        faces = detect_faces_fast(frame)
        if target_embedding is not None:
            target_face, mode, score = _choose_spatial_locked_face(faces, target_embedding, match_threshold, lock_bbox)
        else:
            target_face, mode, score = select_largest_face(faces), 'largest', -1.0
        if target_face is None:
            return frame, False, None, 'miss'
        result = swap_face_fast(frame.copy(), source_face, target_face)
        if use_enhancement and frame_change_ratio(frame, result) >= MIN_SWAP_CHANGE_RATIO:
            result = enhance_face_fast(result, use_gfpgan=True, weight=0.70)
        roi_change = face_roi_change_ratio(frame, result, target_face)
        full_change = frame_change_ratio(frame, result)
        success = roi_change >= 0.003 or full_change >= MIN_SWAP_CHANGE_RATIO
        return result, success, np.asarray(target_face.bbox, dtype=np.float32), mode
    except Exception as e:
        print(f'turbo locked frame skipped: {e}')
        return frame, False, None, 'error'


def _bbox_motion_ratio(previous_bbox, current_bbox):
    if previous_bbox is None or current_bbox is None:
        return 0.0
    previous_bbox = np.asarray(previous_bbox, dtype=np.float32)
    current_bbox = np.asarray(current_bbox, dtype=np.float32)
    prev_center = (previous_bbox[:2] + previous_bbox[2:]) * 0.5
    curr_center = (current_bbox[:2] + current_bbox[2:]) * 0.5
    prev_size = np.maximum(previous_bbox[2:] - previous_bbox[:2], 1.0)
    curr_size = np.maximum(current_bbox[2:] - current_bbox[:2], 1.0)
    center_motion = float(np.linalg.norm(curr_center - prev_center) / (np.linalg.norm(prev_size) + 1e-6))
    scale_motion = float(np.max(np.abs(curr_size / prev_size - 1.0)))
    return max(center_motion, scale_motion)


def _motion_aligned_cached_face(current_frame, cached_swapped, cached_source_bbox, current_bbox):
    """Move the last swapped face patch onto the current detected face box.

    The old v26 cache pasted the face at the previous bbox. During head/body motion,
    that caused visible shake/lag. This version extracts the cached face patch from
    the previous bbox, resizes it, and blends it into the current bbox.
    """
    if cached_swapped is None or cached_source_bbox is None or current_bbox is None:
        return current_frame

    h, w = current_frame.shape[:2]
    src = _clip_bbox(cached_source_bbox, w, h)
    dst = _clip_bbox(current_bbox, w, h)
    if src is None or dst is None:
        return current_frame

    sx1, sy1, sx2, sy2 = src
    dx1, dy1, dx2, dy2 = dst
    patch = cached_swapped[sy1:sy2, sx1:sx2]
    if patch.size == 0:
        return current_frame

    dst_w = dx2 - dx1
    dst_h = dy2 - dy1
    patch = cv2.resize(patch, (dst_w, dst_h), interpolation=cv2.INTER_LINEAR)

    result = current_frame.copy()
    mask = np.zeros((dst_h, dst_w), dtype=np.uint8)
    cv2.ellipse(
        mask,
        (dst_w // 2, dst_h // 2),
        (max(1, dst_w // 2), max(1, dst_h // 2)),
        0, 0, 360, 255, -1,
    )
    blur = max(21, int(min(dst_w, dst_h) * 0.18) | 1)
    mask = cv2.GaussianBlur(mask, (blur, blur), 0).astype(np.float32) / 255.0
    mask3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    roi = result[dy1:dy2, dx1:dx2].astype(np.float32)
    blended = patch.astype(np.float32) * mask3 + roi * (1.0 - mask3)
    result[dy1:dy2, dx1:dx2] = np.clip(blended, 0, 255).astype(np.uint8)
    return result


def _start_frame_reader(cap, queue_size=24):
    read_q = Queue(maxsize=queue_size)

    def _reader():
        index = 0
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                read_q.put((index, frame))
                index += 1
        finally:
            read_q.put(None)

    thread = Thread(target=_reader, daemon=True)
    thread.start()
    return read_q, thread


def _start_frame_writer(out, queue_size=24):
    write_q = Queue(maxsize=queue_size)

    def _writer():
        try:
            while True:
                item = write_q.get()
                if item is None:
                    break
                out.write(item)
        finally:
            out.release()

    thread = Thread(target=_writer, daemon=True)
    thread.start()
    return write_q, thread


def process_video_fast_stable_force_swap(
    source_image_path,
    target_video_path,
    output_path="/content/output_FORCE_REAL_SWAP_FAST.mp4",
    swap_every_n=FAST_SWAP_EVERY_N,
    output_crf=FAST_OUTPUT_CRF,
    output_preset=FAST_OUTPUT_PRESET,
    target_embedding=None,
    target_match_threshold=0.20,
    use_gfpgan_enhancement=FAST_USE_GFPGAN_ENHANCEMENT,
    selected_quality_profile=FAST_QUALITY_PROFILE,
    selected_det_size=FAST_DET_SIZE,
    selected_detect_every_n=1,
    selected_smart_neural=FAST_SMART_NEURAL,
    selected_pose_motion_threshold=0.06,
):
    print("\nStarting v41 reference-stable spatial-lock force swap...")

    source_img = cv2.imread(source_image_path)
    if source_img is None:
        raise ValueError(f"Could not load source image: {source_image_path}")
    source_faces = detect_faces_fast(source_img)
    if not source_faces:
        raise ValueError("No face detected in source image")
    source_face = select_largest_face(source_faces)

    cap = cv2.VideoCapture(target_video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open target video: {target_video_path}")

    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    if not np.isfinite(source_fps) or source_fps <= 0:
        source_fps = FAST_TARGET_OUTPUT_FPS
    fps = source_fps
    output_fps = source_fps
    frame_repeat = 1
    if source_fps < FAST_MIN_OUTPUT_FPS:
        output_fps = FAST_TARGET_OUTPUT_FPS
        frame_repeat = max(1, int(round(output_fps / max(source_fps, 1e-6))))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"   Video: {width}x{height}, source {source_fps:.2f} FPS, output {output_fps:.2f} FPS, {total_frames} frames")
    if frame_repeat > 1:
        print(f"   Low-FPS source detected; each processed frame will be written {frame_repeat}x to preserve duration.")
    stable_every_frame = swap_every_n <= 1
    selected_detect_every_n = max(1, int(selected_detect_every_n or 1))
    selected_smart_neural = bool(selected_smart_neural)
    selected_pose_motion_threshold = float(selected_pose_motion_threshold or 0.06)
    print(f"   Neural swap every: {swap_every_n} frame(s)")
    print(f"   Face detect every: {selected_detect_every_n} frame(s)")
    print(f"   Motion refresh threshold: {selected_pose_motion_threshold:.3f}")
    print(f"   Stable fast path: {'ON' if stable_every_frame else 'OFF'}")
    print(f"   Single-person lock: {'ON' if target_embedding is not None else 'OFF'}")
    print(f"   Selected mode: {_FAST_PROFILES.get(selected_quality_profile, {}).get('label', selected_quality_profile)}")
    print(f"   GFPGAN restoration: {'ON' if use_gfpgan_enhancement else 'OFF'}")

    warm_ok, warm_frame = cap.read()
    if warm_ok:
        print("   Warming up inswapper on real target frame...")
        for _ in range(3):
            with torch.amp.autocast('cuda', enabled=USE_FP16 and torch.cuda.is_available()):
                _ = process_frame_optimized(
                    warm_frame,
                    source_face,
                    use_enhancement=use_gfpgan_enhancement,
                    target_embedding=target_embedding,
                    match_threshold=target_match_threshold,
                    force_largest=(target_embedding is None),
                )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    else:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    out, temp_output, intermediate_label = _open_high_quality_writer(output_path, output_fps, width, height)
    read_q, reader_thread = _start_frame_reader(cap)
    write_q, writer_thread = _start_frame_writer(out)
    processed = 0
    changed = 0
    neural_swaps = 0
    cached_frames = 0
    frame_times = []
    last_swapped = None
    last_neural_bbox = None
    smoothed_bbox = None
    previous_frame = None
    tracked_bbox = None
    optical_flow_updates = 0
    spatial_lock_swaps = 0
    pose_forced_swaps = 0
    motion_forced_swaps = 0

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    pbar = tqdm(total=total_frames, desc="v41 stable spatial lock", dynamic_ncols=True, leave=True, mininterval=0.5)
    try:
        while processed < total_frames:
            item = read_q.get()
            if item is None:
                break
            _, frame = item
            start = time.perf_counter()
            current_bbox = None
            motion_forces_neural = False
            do_neural = stable_every_frame or (processed % swap_every_n == 0) or last_swapped is None
            needs_bbox_detect = stable_every_frame or do_neural or (processed % selected_detect_every_n == 0) or (selected_smart_neural and processed % selected_detect_every_n == 0)
            if needs_bbox_detect:
                current_bbox = _target_face_bbox(frame, target_embedding, target_match_threshold)
                if selected_smart_neural and not stable_every_frame:
                    motion_score = _face_roi_motion_score(previous_frame, frame, current_bbox)
                    motion_forces_neural = (
                        last_swapped is not None
                        and motion_score >= FAST_MOTION_NEURAL_THRESHOLD
                    )
                    do_neural = do_neural or motion_forces_neural
            elif last_neural_bbox is not None:
                previous_tracked_bbox = tracked_bbox if tracked_bbox is not None else last_neural_bbox
                tracked = _track_bbox_optical_flow(previous_frame, frame, previous_tracked_bbox)
                if tracked is not None:
                    current_bbox = tracked
                    tracked_bbox = tracked
                    optical_flow_updates += 1
                    pose_motion = _bbox_motion_ratio(previous_tracked_bbox, tracked)
                    if selected_smart_neural and pose_motion >= selected_pose_motion_threshold:
                        do_neural = True
                        motion_forces_neural = True
                        pose_forced_swaps += 1
                else:
                    current_bbox = tracked_bbox if tracked_bbox is not None else last_neural_bbox
                    if selected_smart_neural:
                        do_neural = True
                        motion_forces_neural = True
                        pose_forced_swaps += 1

            if do_neural:
                with torch.amp.autocast('cuda', enabled=USE_FP16 and torch.cuda.is_available()):
                    result, success, selected_bbox, select_mode = _process_frame_turbo_locked(
                        frame,
                        source_face,
                        target_embedding=target_embedding,
                        match_threshold=target_match_threshold,
                        lock_bbox=current_bbox if current_bbox is not None else tracked_bbox,
                        use_enhancement=use_gfpgan_enhancement,
                    )
                if selected_bbox is not None:
                    current_bbox = selected_bbox
                if select_mode == 'spatial_lock':
                    spatial_lock_swaps += 1
                if not stable_every_frame:
                    smoothed_bbox = _smooth_bbox(smoothed_bbox, current_bbox, alpha=0.92)
                if success:
                    if not stable_every_frame:
                        if current_bbox is None:
                            current_bbox = last_neural_bbox
                        last_swapped = result.copy()
                        last_neural_bbox = smoothed_bbox.copy() if smoothed_bbox is not None else current_bbox
                        tracked_bbox = last_neural_bbox.copy() if last_neural_bbox is not None else tracked_bbox
                    changed += 1
                neural_swaps += 1
                if motion_forces_neural:
                    motion_forced_swaps += 1
            else:
                smoothed_bbox = _smooth_bbox(smoothed_bbox, current_bbox, alpha=0.90)
                result = _motion_aligned_cached_face(frame, last_swapped, last_neural_bbox, smoothed_bbox)
                success = last_swapped is not None and smoothed_bbox is not None
                cached_frames += 1
                if success:
                    if smoothed_bbox is not None:
                        tracked_bbox = smoothed_bbox.copy()
                    changed += 1

            for _ in range(frame_repeat):
                write_q.put(result)
            if not stable_every_frame:
                previous_frame = frame.copy()
            elapsed = time.perf_counter() - start
            frame_times.append(elapsed)
            processed += 1
            if processed % FAST_PROGRESS_UPDATE_EVERY == 0 or processed == total_frames:
                pbar.set_postfix({"FPS": f"{1.0 / max(np.mean(frame_times[-30:]), 1e-6):.1f}", "N": neural_swaps, "C": cached_frames})
            pbar.update(1)

            if processed % 90 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
    finally:
        pbar.close()
        write_q.put(None)
        writer_thread.join()
        reader_thread.join(timeout=2.0)
        cap.release()

    print("\nCreating final MP4...")
    mux_ok = _mux_high_quality_video(
        temp_output=temp_output,
        target_video_path=target_video_path,
        output_path=output_path,
        crf=output_crf,
        preset=output_preset,
    )
    if mux_ok:
        try:
            os.remove(temp_output)
        except Exception:
            pass
    elif os.path.exists(temp_output):
        os.rename(temp_output, output_path)

    avg_fps = 1.0 / np.mean(frame_times) if frame_times else 0
    stats = {
        "output_path": output_path,
        "total_frames": processed,
        "changed_frames": changed,
        "neural_swaps": neural_swaps,
        "cached_frames": cached_frames,
        "motion_forced_swaps": motion_forced_swaps,
        "pose_forced_swaps": pose_forced_swaps,
        "spatial_lock_swaps": spatial_lock_swaps,
        "optical_flow_updates": optical_flow_updates,
        "swap_every_n": swap_every_n,
        "detect_every_n": selected_detect_every_n,
        "single_person_locked": target_embedding is not None,
        "target_match_threshold": target_match_threshold,
        "smart_neural": selected_smart_neural,
        "stable_fast_path": stable_every_frame,
        "threaded_pipeline": True,
        "nvenc_preferred": True,
        "motion_threshold": FAST_MOTION_NEURAL_THRESHOLD,
        "pose_motion_threshold": selected_pose_motion_threshold,
        "avg_fps": avg_fps,
        "avg_frame_time_ms": float(np.mean(frame_times) * 1000) if frame_times else 0,
        "output_crf": output_crf,
        "output_preset": output_preset,
        "intermediate": intermediate_label,
        "cache_mode": "motion_aligned_cache" if swap_every_n > 1 else "stable_every_frame_fast_path",
        "quality_profile": selected_quality_profile,
        "quality_mode_label": _FAST_PROFILES.get(selected_quality_profile, {}).get("label", selected_quality_profile),
        "det_size": selected_det_size,
        "gfpgan_enhancement": use_gfpgan_enhancement,
        "source_fps": source_fps,
        "output_fps": output_fps,
        "frame_repeat": frame_repeat,
        "fps_target": "15-25 FPS" if selected_quality_profile == "TURBO_15_25_FPS" else "quality-dependent",
    }

    print("\n" + "=" * 70)
    print("STABLE FAST PROCESSING COMPLETE")
    print("=" * 70)
    print(f"   Output: {output_path}")
    print(f"   Average processing FPS: {avg_fps:.1f}")
    print(f"   Output FPS: {output_fps:.2f} (source {source_fps:.2f}, repeat {frame_repeat}x)")
    print(f"   Neural frames: {neural_swaps}, cached frames: {cached_frames}, optical-flow tracked: {optical_flow_updates}, spatial-lock swaps: {spatial_lock_swaps}, pose-forced neural: {pose_forced_swaps}, motion-forced neural: {motion_forced_swaps}")
    print(f"   Changed frames: {changed}/{processed} ({100 * changed / max(processed, 1):.1f}%)")
    if selected_quality_profile == "TURBO_15_25_FPS" and avg_fps >= 15:
        print(f"   TURBO TARGET ACHIEVED ({avg_fps:.1f} FPS >= 15 FPS)")
    elif selected_quality_profile == "TURBO_15_25_FPS":
        print("   Turbo target not reached. This runtime/video is bottlenecked by inswapper, decoding, or full-resolution patch blending.")
    elif avg_fps >= 15:
        print(f"   TARGET ACHIEVED ({avg_fps:.1f} FPS >= 15 FPS)")
    else:
        print("   To go faster choose Turbo 15-25 FPS mode; for best detail choose High Quality.")
    print("=" * 70)
    return stats



# ===================== UPLOADED STYLE UI (v31 stable no-drift) =====================
from IPython.display import display, HTML, clear_output, Video
from google.colab import files, output
import ipywidgets as widgets
from pathlib import Path
import base64
import os

UI_UPLOAD_DIR = Path('/content/ui_uploads')
UI_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
UI_PREVIEW_DIR = Path('/content/ui_preview')
UI_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

_ui_state = {'source_path': None, 'video_path': None, 'source_name': None, 'video_name': None, 'video_thumb': None, 'target_embedding': None, 'target_face_index': 0}

_ui_css = """
<style>
.face-swap-card, .face-preview-card { min-height: 275px; border-radius: 8px; border: 1px solid #2b2940; background: #1e1e2c; padding: 14px; position: relative; overflow: hidden; color: white; font-family: Inter, Arial, sans-serif; }
.face-swap-card.face-clickable { cursor: pointer; transition: border-color .15s ease, transform .15s ease; }
.face-swap-card.face-clickable:hover { border-color: #8b6dff; transform: translateY(-1px); }
.face-preview-card { min-height: 210px; }
.face-step { width: 24px; height: 24px; border-radius: 50%; display: inline-flex; align-items: center; justify-content: center; background: #8973ff; color: #141126; font-weight: 800; margin-right: 14px; }
.face-tabs { display: inline-flex; gap: 4px; padding: 4px; border: 1px solid #30304a; border-radius: 999px; background: #171827; font-size: 12px; }
.face-tab-active { background: #8b6dff; color: white; border-radius: 999px; padding: 8px 14px; font-weight: 700; }
.face-tab { color: #f5f3ff; padding: 8px 14px; font-weight: 700; }
.face-drop-copy { text-align: center; margin: 56px 12px 8px; }
.face-icon { font-size: 24px; line-height: 1; margin-bottom: 14px; }
.face-title { font-size: 16px; font-weight: 700; margin-bottom: 6px; }
.face-subtitle { color: #a7a2c8; font-size: 13px; line-height: 1.35; }
.face-media { width: 100%; height: 205px; object-fit: contain; border-radius: 8px; background: #11101d; margin-top: 16px; }
.face-preview-grid { display: grid; grid-template-columns: repeat(2, minmax(180px, 1fr)); gap: 10px; }
.face-preview-media { width: 100%; height: 150px; object-fit: contain; border-radius: 8px; background: #11101d; margin-top: 10px; }
.face-caption { color: #c9c3ef; font-size: 12px; margin-top: 8px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.face-quota { color: #13f2a3; background: rgba(19, 242, 163, 0.12); border-radius: 999px; padding: 7px 12px; font-size: 13px; font-weight: 800; }
.face-status-line { max-width: 1040px; color: #d8d3ff; background: #171827; border: 1px solid #2b2940; border-radius: 8px; padding: 9px 12px; margin: 10px 0; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; font-family: Inter, Arial, sans-serif; }
.face-upload-line { max-width: 1040px; color: #13f2a3; background: rgba(19, 242, 163, 0.10); border: 1px solid rgba(19, 242, 163, 0.25); border-radius: 8px; padding: 9px 12px; margin: 8px 0; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; font-family: Inter, Arial, sans-serif; }
</style>
"""

face_tabs = "<span class='face-tab-active'>Face</span><span class='face-tab'>Head</span><span class='face-tab'>Body</span>"
person_tabs = "<span class='face-tab-active'>Single</span><span class='face-tab'>Multi</span>"
base_box = widgets.HTML()
face_box = widgets.HTML()
preview_box = widgets.HTML()
status_line = widgets.HTML('<div class="face-status-line">Use Upload Video and Upload Image, then click Swap Faces.</div>')
upload_line = widgets.HTML('<div class="face-upload-line">Upload status: waiting for file selection.</div>')
upload_progress = widgets.IntProgress(value=0, min=0, max=100, description='Save:', bar_style='info', layout=widgets.Layout(width='360px'))
upload_progress.layout.display = 'none'
status_out = widgets.Output()
video_uploader = None
face_uploader = None
upload_controls = widgets.Output(layout=widgets.Layout(max_width='1040px', margin='12px 0 0 0'))
quality_mode = widgets.Dropdown(
    options=[
        ('Turbo 15-25 FPS - target speed', 'TURBO_15_25_FPS'),
        ('Fast Face Only - faster', 'FAST_FACE_ONLY'),
        ('Better Face Coverage - balanced', 'BETTER_FACE_COVERAGE'),
        ('High Quality - slower', 'HIGH_QUALITY'),
    ],
    value='TURBO_15_25_FPS',
    description='Mode:',
    layout=widgets.Layout(width='360px'),
    style={'description_width': '60px'},
)
swap_button = widgets.Button(description='Swap Faces', icon='magic', layout=widgets.Layout(width='150px', height='42px'))
swap_button.style.button_color = '#6c18ff'


def _set_status(text):
    status_line.value = f'<div class="face-status-line">{text}</div>'


def _set_upload_status(text):
    upload_line.value = f'<div class="face-upload-line">{text}</div>'


def _set_upload_progress(percent=None, label='Save:'):
    if percent is None:
        upload_progress.layout.display = 'none'
        upload_progress.value = 0
        upload_progress.bar_style = 'info'
        return
    pct = int(max(0, min(100, percent)))
    upload_progress.description = label
    upload_progress.value = pct
    upload_progress.bar_style = 'success' if pct >= 100 else 'info'
    upload_progress.layout.display = ''


def _format_size(num_bytes):
    value = float(num_bytes or 0)
    for unit in ['B', 'KB', 'MB', 'GB']:
        if value < 1024.0 or unit == 'GB':
            return f'{value:.1f} {unit}'
        value /= 1024.0


def _data_url(content, mime):
    return 'data:' + mime + ';base64,' + base64.b64encode(content).decode('ascii')


def _safe_name(name):
    return ''.join(c if c.isalnum() or c in '._-' else '_' for c in str(name))


def _remove_file(path):
    if path and os.path.exists(path):
        try:
            os.remove(path)
        except Exception as e:
            print(f'Could not remove old file {path}: {e}')


def _cleanup_upload_kind(kind):
    if kind == 'video':
        for key in ['video_path', 'video_thumb']:
            _remove_file(_ui_state.get(key))
        _ui_state.update({'video_path': None, 'video_name': None, 'video_thumb': None, 'target_embedding': None, 'target_face_index': 0})
        _set_preview_box()
        for old in UI_UPLOAD_DIR.glob('target_video_*'):
            _remove_file(str(old))
        for old in UI_PREVIEW_DIR.glob('*'):
            _remove_file(str(old))
        _remove_file('/content/output_FORCE_REAL_SWAP_FAST.mp4')
    elif kind == 'source':
        _remove_file(_ui_state.get('source_path'))
        _ui_state.update({'source_path': None, 'source_name': None, 'target_embedding': None})
        _set_preview_box()
        for old in UI_UPLOAD_DIR.glob('source_face_*'):
            _remove_file(str(old))


def _upload_html():
    return """
    <div id="fs-upload-wrap" style="max-width:1040px;display:flex;gap:12px;flex-wrap:wrap;align-items:flex-start;font-family:Inter,Arial,sans-serif;">
      <div style="background:#171827;border:1px solid #2b2940;border-radius:8px;padding:10px 12px;color:white;min-width:260px;">
        <div style="font-weight:800;margin-bottom:8px;">Upload Video</div>
        <input id="fs-video-input" type="file" accept="video/*" style="color:#d8d3ff;max-width:230px;" />
        <div id="fs-video-status" style="color:#13f2a3;font-size:12px;margin-top:8px;">Waiting for video.</div>
        <progress id="fs-video-progress" max="100" value="0" style="width:230px;margin-top:8px;"></progress>
      </div>
      <div style="background:#171827;border:1px solid #2b2940;border-radius:8px;padding:10px 12px;color:white;min-width:260px;">
        <div style="font-weight:800;margin-bottom:8px;">Upload Face Image</div>
        <input id="fs-face-input" type="file" accept="image/*" style="color:#d8d3ff;max-width:230px;" />
        <div id="fs-face-status" style="color:#13f2a3;font-size:12px;margin-top:8px;">Waiting for image.</div>
        <progress id="fs-face-progress" max="100" value="0" style="width:230px;margin-top:8px;"></progress>
      </div>
    </div>
    <script>
    (() => {
      const chunkSize = 512 * 1024;
      const toBase64 = (buffer) => {
        let binary = '';
        const bytes = new Uint8Array(buffer);
        const step = 0x8000;
        for (let i = 0; i < bytes.length; i += step) {
          binary += String.fromCharCode.apply(null, bytes.subarray(i, i + step));
        }
        return btoa(binary);
      };
      const setStatus = (kind, text, pct) => {
        const status = document.getElementById(kind === 'video' ? 'fs-video-status' : 'fs-face-status');
        const progress = document.getElementById(kind === 'video' ? 'fs-video-progress' : 'fs-face-progress');
        if (status) status.textContent = text;
        if (progress && Number.isFinite(pct)) progress.value = pct;
      };
      const uploadFile = async (kind, file) => {
        if (!file) return;
        const totalChunks = Math.max(1, Math.ceil(file.size / chunkSize));
        setStatus(kind, 'Starting 0% | ' + file.name, 0);
        for (let index = 0; index < totalChunks; index++) {
          const start = index * chunkSize;
          const end = Math.min(file.size, start + chunkSize);
          const chunk = await file.slice(start, end).arrayBuffer();
          const b64 = toBase64(chunk);
          await google.colab.kernel.invokeFunction(
            'face_swap_upload.receive_chunk',
            [kind, file.name, file.type || '', b64, index, totalChunks, file.size],
            {}
          );
          const pct = Math.round(((index + 1) / totalChunks) * 100);
          setStatus(kind, 'Uploaded ' + pct + '% | ' + file.name, pct);
        }
        setStatus(kind, 'Complete 100% | ' + file.name, 100);
        const input = document.getElementById(kind === 'video' ? 'fs-video-input' : 'fs-face-input');
        if (input) input.value = '';
      };
      const videoInput = document.getElementById('fs-video-input');
      const faceInput = document.getElementById('fs-face-input');
      if (videoInput) videoInput.onchange = (e) => uploadFile('video', e.target.files[0]).catch(err => setStatus('video', 'ERROR: ' + err.message, 0));
      if (faceInput) faceInput.onchange = (e) => uploadFile('source', e.target.files[0]).catch(err => setStatus('source', 'ERROR: ' + err.message, 0));
    })();
    </script>
    """


def _refresh_upload_widgets():
    """Render a chunked Colab uploader with real browser-to-runtime progress."""
    with upload_controls:
        clear_output(wait=True)
        display(HTML(_upload_html()))


def _reset_upload_controls():
    _set_upload_progress(None)
    _refresh_upload_widgets()


_upload_sessions = {}


def _receive_upload_chunk(kind, name, mime, chunk_b64, index, total_chunks, total_bytes):
    kind = 'video' if kind == 'video' else 'source'
    prefix = 'target_video' if kind == 'video' else 'source_face'
    accept_note = 'base video' if kind == 'video' else 'face photo'
    index = int(index)
    total_chunks = max(1, int(total_chunks))
    total_bytes = int(total_bytes)
    if index == 0:
        _cleanup_upload_kind('video' if kind == 'video' else 'source')
        path = UI_UPLOAD_DIR / f'{prefix}_{int(time.time())}_{_safe_name(name)}'
        with open(path, 'wb'):
            pass
        _upload_sessions[kind] = {'path': str(path), 'name': name, 'total_bytes': total_bytes, 'written': 0}
        _set_upload_progress(0, 'Upload:')
        _set_upload_status(f'{accept_note}: uploading 0% | {name} | {_format_size(total_bytes)}')
    session = _upload_sessions.get(kind)
    if not session:
        raise RuntimeError('Upload session was not initialized. Select the file again.')
    chunk = base64.b64decode(chunk_b64)
    with open(session['path'], 'ab') as f:
        f.write(chunk)
    session['written'] += len(chunk)
    pct = int(((index + 1) / total_chunks) * 100)
    _set_upload_progress(pct, 'Upload:')
    _set_upload_status(f'{accept_note}: uploading {pct}% | {name} | {_format_size(total_bytes)}')
    if index + 1 >= total_chunks:
        saved_size = os.path.getsize(session['path']) if os.path.exists(session['path']) else -1
        if saved_size != total_bytes:
            raise IOError(f'Saved file size mismatch: expected {total_bytes}, got {saved_size}')
        if kind == 'video':
            _ui_state['video_path'] = session['path']
            _ui_state['video_name'] = name
            _ui_state['video_thumb'] = _video_thumbnail(session['path'])
            _ui_state['target_embedding'] = None
            _set_status(f'Base video ready | {name}. Previous video was removed.')
        else:
            _ui_state['source_path'] = session['path']
            _ui_state['source_name'] = name
            _ui_state['target_embedding'] = None
            _set_status(f'Face photo ready | {name}. Previous face photo was removed.')
        _render_upload_cards()
        _set_upload_progress(100, 'Done:')
        _set_upload_status(f'{accept_note}: upload complete 100% | {name} | {_format_size(total_bytes)}')
        _upload_sessions.pop(kind, None)
    return {'ok': True, 'percent': pct}


output.register_callback('face_swap_upload.receive_chunk', _receive_upload_chunk)


def _image_data_url(path):
    with open(path, 'rb') as f:
        return _data_url(f.read(), 'image/jpeg' if path.lower().endswith(('.jpg', '.jpeg')) else 'image/png')


def _video_thumbnail(video_path):
    thumb_path = str(UI_PREVIEW_DIR / 'uploaded_video_thumb.jpg')
    cap = cv2.VideoCapture(video_path)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    cv2.imwrite(thumb_path, frame)
    return thumb_path


def _empty_card(step, title, subtitle, tabs):
    return f"""
    <div class='face-swap-card'>
      <div><span class='face-step'>{step}</span><span class='face-tabs'>{tabs}</span></div>
      <div class='face-drop-copy'><div class='face-icon'>*</div><div class='face-title'>{title}</div><div class='face-subtitle'>{subtitle}</div></div>
    </div>
    """


def _render_upload_cards():
    if _ui_state.get('video_path') and _ui_state.get('video_thumb'):
        url = _image_data_url(_ui_state['video_thumb'])
        base_box.value = f"<div class='face-swap-card'><div><span class='face-step'>1</span><span class='face-tabs'>{face_tabs}</span></div><img class='face-media' src='{url}'/><div class='face-caption'>{_ui_state['video_name']}</div></div>"
    else:
        base_box.value = _empty_card(1, 'Upload base video', 'Use the browser upload panel below.', face_tabs)
    if _ui_state.get('source_path'):
        url = _image_data_url(_ui_state['source_path'])
        face_box.value = f"<div class='face-swap-card'><div><span class='face-step'>2</span><span class='face-tabs'>{person_tabs}</span></div><img class='face-media' src='{url}'/><div class='face-caption'>{_ui_state['source_name']}</div></div>"
    else:
        face_box.value = _empty_card(2, 'Upload face photo', 'Use the browser upload panel below.', person_tabs)


def _extract_widget_upload(upload_value):
    if not upload_value:
        return None, None
    if isinstance(upload_value, dict):
        name = next(iter(upload_value.keys()))
        item = upload_value[name]
    else:
        item = upload_value[0]
        name = item.get('name', 'uploaded_file')
    content = item.get('content')
    if isinstance(content, memoryview):
        content = content.tobytes()
    return name, bytes(content)


def _save_widget_upload(upload_value, prefix, accept_note):
    name, content = _extract_widget_upload(upload_value)
    if not name or content is None:
        _set_upload_progress(None)
        _set_upload_status(f'{accept_note}: no file selected.')
        _set_status('No file selected.')
        return None, None
    total_size = len(content)
    _set_upload_progress(0, 'Save:')
    _set_upload_status(f'{accept_note}: file received by Colab. Saving to runtime 0% | {name} | {_format_size(total_size)}')
    path = UI_UPLOAD_DIR / f'{prefix}_{int(time.time())}_{_safe_name(name)}'
    chunk_size = max(1024 * 1024, total_size // 100 if total_size else 1024 * 1024)
    written = 0
    with open(path, 'wb') as f:
        for offset in range(0, total_size, chunk_size):
            chunk = content[offset:offset + chunk_size]
            f.write(chunk)
            written += len(chunk)
            pct = 100 if total_size == 0 else int((written / total_size) * 100)
            _set_upload_progress(pct, 'Save:')
            _set_upload_status(f'{accept_note}: saving to runtime {pct}% | {name} | {_format_size(total_size)}')
    saved_size = os.path.getsize(path) if os.path.exists(path) else -1
    if saved_size != total_size:
        raise IOError(f'Saved file size mismatch: expected {total_size}, got {saved_size}')
    _set_upload_progress(100, 'Saved:')
    _set_upload_status(f'{accept_note}: saved 100% | {name} | {_format_size(total_size)}')
    return str(path), name


def _handle_video_upload(change):
    if change.get('name') != 'value' or not change.get('new'):
        return
    with status_out:
        clear_output()
        try:
            _set_status('Removing previous base video and saving the new one...')
            _cleanup_upload_kind('video')
            path, name = _save_widget_upload(change['new'], 'target_video', 'base video')
            if not path:
                return
            _ui_state['video_path'] = path
            _ui_state['video_name'] = name
            _ui_state['video_thumb'] = _video_thumbnail(path)
            _ui_state['target_embedding'] = None
            _render_upload_cards()
            _set_status(f'Base video ready | {name}. Previous video was removed; upload another anytime without rerunning this cell.')
        except Exception as e:
            _set_status(f'ERROR uploading video: {e}')
            print(f'ERROR uploading video: {e}')
        finally:
            _reset_upload_controls()


def _handle_face_upload(change):
    if change.get('name') != 'value' or not change.get('new'):
        return
    with status_out:
        clear_output()
        try:
            _set_status('Removing previous face photo and saving the new one...')
            _cleanup_upload_kind('source')
            path, name = _save_widget_upload(change['new'], 'source_face', 'face photo')
            if not path:
                return
            _ui_state['source_path'] = path
            _ui_state['source_name'] = name
            _render_upload_cards()
            _set_status(f'Face photo ready | {name}. Previous face photo was removed; upload another anytime without rerunning this cell.')
        except Exception as e:
            _set_status(f'ERROR uploading face photo: {e}')
            print(f'ERROR uploading face photo: {e}')
        finally:
            _reset_upload_controls()


def _set_preview_box(before_path=None, after_path=None):
    if not before_path or not after_path:
        preview_box.value = "<div class='face-preview-card'><div><span class='face-step'>3</span><span class='face-tabs'><span class='face-tab-active'>Preview</span></span></div><div class='face-drop-copy'><div class='face-icon'>▧</div><div class='face-title'>Swap preview</div><div class='face-subtitle'>Target frame and swapped preview will show here.</div></div></div>"
        return
    before_url = _image_data_url(before_path)
    after_url = _image_data_url(after_path)
    preview_box.value = f"<div class='face-preview-card'><div><span class='face-step'>3</span><span class='face-tabs'><span class='face-tab-active'>Preview</span></span></div><div class='face-preview-grid'><div><img class='face-preview-media' src='{before_url}'/><div class='face-caption'>Target frame</div></div><div><img class='face-preview-media' src='{after_url}'/><div class='face-caption'>Swapped preview</div></div></div></div>"


def _create_swap_preview(source_path, video_path):
    source_img = cv2.imread(source_path)
    source_faces = detect_faces_fast(source_img)
    if not source_faces:
        raise ValueError('No face detected in source image. Use a clearer front-facing source photo.')
    source_face = select_largest_face(source_faces)
    cap = cv2.VideoCapture(video_path)
    preview_frame, preview_faces = None, []
    for _ in range(180):
        ret, frame = cap.read()
        if not ret:
            break
        faces = detect_faces_fast(frame)
        if faces:
            preview_frame, preview_faces = frame, faces
            break
    cap.release()
    if preview_frame is None:
        raise ValueError('No face detected in the first 180 target-video frames.')
    target_face = select_largest_face(preview_faces)
    target_embedding = _face_embedding(target_face)
    if target_embedding is None:
        raise ValueError('Selected target face has no embedding for single-person lock.')
    raw_result = swap_face_fast(preview_frame.copy(), source_face, target_face)
    full_change = frame_change_ratio(preview_frame, raw_result)
    roi_change = face_roi_change_ratio(preview_frame, raw_result, target_face)
    if roi_change < 0.003 and full_change < MIN_SWAP_CHANGE_RATIO:
        raise RuntimeError('Preview did not visibly change inside the face area. Check uploaded files.')
    before_path = str(UI_PREVIEW_DIR / 'target_frame.jpg')
    after_path = str(UI_PREVIEW_DIR / 'swapped_preview.jpg')
    cv2.imwrite(before_path, preview_frame, [cv2.IMWRITE_JPEG_QUALITY, 98])
    cv2.imwrite(after_path, raw_result, [cv2.IMWRITE_JPEG_QUALITY, 98])
    return before_path, after_path, target_embedding


def _print_integrated_benchmark(stats):
    total = int(stats.get('total_frames', 0) or 0)
    avg_fps = float(stats.get('avg_fps', 0) or 0)
    avg_ms = float(stats.get('avg_frame_time_ms', 0) or 0)
    changed = int(stats.get('changed_frames', 0) or 0)
    print('\nBENCHMARK TEST / RUN SUMMARY (inside UI)')
    print('=' * 70)
    print('This uses the actual processing run, so no extra benchmark pass is executed.')
    print(f'   Processing FPS: {avg_fps:.1f} ({avg_ms:.1f} ms/frame)')
    print(f'   Frames processed: {total}')
    print(f'   Changed frames: {changed}/{max(total, 1)} ({100 * changed / max(total, 1):.1f}%)')
    print(f"   Mode: {stats.get('quality_mode_label', stats.get('quality_profile', FAST_QUALITY_PROFILE))} | Detector: {stats.get('det_size', FAST_DET_SIZE)} | CRF/CQ: {stats.get('output_crf', FAST_OUTPUT_CRF)} | GFPGAN: {stats.get('gfpgan_enhancement', False)}")
    print(f"   Output FPS: {stats.get('output_fps', 0):.2f} (source {stats.get('source_fps', 0):.2f}, repeat {stats.get('frame_repeat', 1)}x)")
    print('=' * 70)


def _run_ui_swap(_):
    global source_path, video_path, output_path, stats
    swap_button.disabled = True
    with status_out:
        clear_output()
        try:
            source_path = _ui_state.get('source_path')
            video_path = _ui_state.get('video_path')
            if not source_path or not video_path:
                _set_status('Please upload both files first with the Upload Video and Upload Image buttons. Single mode locks the target from preview.')
                return
            _set_status('Creating target and swapped preview...')
            before_path, after_path, target_embedding = _create_swap_preview(source_path, video_path)
            _ui_state['target_embedding'] = target_embedding
            _set_preview_box(before_path, after_path)
            selected_mode = quality_mode.value
            selected_profile = _FAST_PROFILES.get(selected_mode, _FAST_PROFILES['TURBO_15_25_FPS'])
            _set_status(f"Preview ready | Single target locked | Running {selected_profile['label']} mode")
            try:
                face_app.prepare(ctx_id=0 if torch.cuda.is_available() else -1, det_size=selected_profile['det_size'])
                print(f"Detector prepared for {selected_profile['label']}: {selected_profile['det_size']}")
            except Exception as e:
                print(f"Detector re-prepare skipped: {e}")
            output_path = '/content/output_FORCE_REAL_SWAP_FAST.mp4'
            if os.path.exists(output_path):
                os.remove(output_path)
            stats = process_video_fast_stable_force_swap(
                source_image_path=source_path,
                target_video_path=video_path,
                output_path=output_path,
                swap_every_n=selected_profile['swap_every_n'],
                output_crf=selected_profile['crf'],
                output_preset=selected_profile['preset'],
                target_embedding=_ui_state.get('target_embedding'),
                target_match_threshold=selected_profile['match_threshold'],
                use_gfpgan_enhancement=selected_profile['gfpgan'],
                selected_quality_profile=selected_mode,
                selected_det_size=selected_profile['det_size'],
                selected_detect_every_n=selected_profile.get('detect_every_n', 1),
                selected_smart_neural=selected_profile.get('smart', False),
                selected_pose_motion_threshold=selected_profile.get('pose_motion_threshold', 0.06),
            )
            if stats.get('changed_frames', 0) == 0:
                raise RuntimeError('No frames changed. Check the uploaded face photo and base video.')
            _set_status(f"Done | FPS: {stats.get('avg_fps', 0):.1f} | Changed: {stats.get('changed_frames', 0)}/{stats.get('total_frames', 0)} | Downloading output")
            _print_integrated_benchmark(stats)
            print('\nDOWNLOAD RESULT (inside UI)')
            print('=' * 70)
            print(f'   Output: {output_path}')
            display(Video(output_path, width=640))
            files.download(output_path)
        except Exception as e:
            _set_status(f'ERROR: {e}')
            print(f'ERROR: {e}')
        finally:
            swap_button.disabled = False

_refresh_upload_widgets()
swap_button.on_click(_run_ui_swap)
_render_upload_cards()
_set_preview_box()

display(HTML(_ui_css))
display(widgets.VBox([
    widgets.HBox([base_box, face_box], layout=widgets.Layout(max_width='1040px', gap='12px')),
    upload_controls,
    widgets.HBox([preview_box], layout=widgets.Layout(max_width='1040px', margin='12px 0 0 0')),
    widgets.HBox([quality_mode, widgets.HTML("<span class='face-quota'>v41 reference-stable spatial lock</span>"), swap_button], layout=widgets.Layout(max_width='1040px', justify_content='space-between', align_items='center', margin='12px 0 0 0')),
    upload_line,
    upload_progress,
    status_line,
    status_out,
]))


## Batch image face swap (v44)

This optional workspace adds high-quality still-image processing without changing the existing video workflow.

- Upload **one source face** and **many target images**.
- Replace the **largest face only** or **every detected face** in each target image.
- **High Quality** uses the same InsightFace swap model and GFPGAN restoration path as the video High Quality mode.
- Results are saved as lossless PNG files and downloaded together as a ZIP when more than one image is processed.
- Uploads use reliable 2 MB chunks with separate **current-file** and **overall** percentage bars. Additional target selections append to the ready list.

Run the notebook from the top first so the models are loaded, then use the panel below. Use face-swap results only with the permission of the people shown.


In [ ]:
# ===================== BATCH IMAGE FACE SWAP (v44) =====================
# One source face -> many target images. Uses the same swapper and optional
# GFPGAN enhancement as the High Quality video profile, with lossless PNG output.
import base64
import html
import os
import re
import shutil
import time
import zipfile
from pathlib import Path

import cv2
import ipywidgets as widgets
import numpy as np
import torch
from IPython.display import HTML, clear_output, display
from PIL import Image as PILImage, ImageOps
from google.colab import files, output

print("BATCH IMAGE FACE SWAP (v44 - RELIABLE CHUNK UPLOAD + PROGRESS)")
print("=" * 70)

_batch_required = [
    "face_app", "swapper", "detect_faces_fast", "select_largest_face",
    "swap_face_fast", "enhance_face_fast", "face_roi_change_ratio",
]
_batch_missing = [name for name in _batch_required if name not in globals()]
if _batch_missing:
    raise RuntimeError(
        "Run the notebook from the top before opening Batch Image mode. "
        f"Missing: {_batch_missing}"
    )

BATCH_ROOT = Path("/content/batch_image_face_swap")
BATCH_SOURCE_DIR = BATCH_ROOT / "source"
BATCH_TARGET_DIR = BATCH_ROOT / "targets"
BATCH_OUTPUT_DIR = BATCH_ROOT / "outputs"
BATCH_PREVIEW_DIR = BATCH_ROOT / "previews"
for _folder in (BATCH_SOURCE_DIR, BATCH_TARGET_DIR, BATCH_OUTPUT_DIR, BATCH_PREVIEW_DIR):
    _folder.mkdir(parents=True, exist_ok=True)

_batch_state = {
    "source_path": None,
    "source_name": None,
    "target_paths": [],
    "target_names": [],
    "results": [],
}

_BATCH_PROFILES = {
    "HIGH_QUALITY": {
        "label": "High Quality - video HQ model + GFPGAN",
        "det_size": (768, 768),
        "gfpgan": True,
        "gfpgan_weight": 0.70,
        "extension": ".png",
    },
    "NATURAL_HQ": {
        "label": "Natural HQ - no restoration",
        "det_size": (768, 768),
        "gfpgan": False,
        "gfpgan_weight": 0.0,
        "extension": ".png",
    },
    "FAST_BATCH": {
        "label": "Fast Batch - smaller detector",
        "det_size": (512, 512),
        "gfpgan": False,
        "gfpgan_weight": 0.0,
        "extension": ".png",
    },
}


def _batch_safe_name(name):
    name = Path(str(name or "image")).name
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(name).stem).strip("._") or "image"
    suffix = Path(name).suffix.lower() or ".png"
    return stem[:80] + suffix[:10]


def _batch_upload_items(value):
    """Support ipywidgets 7 (dict) and 8 (tuple/list) upload values."""
    if not value:
        return []
    if isinstance(value, dict):
        items = []
        for name, item in value.items():
            record = dict(item)
            record.setdefault("name", name)
            items.append(record)
        return items
    return list(value)


def _batch_bytes(content):
    if isinstance(content, memoryview):
        return content.tobytes()
    return bytes(content)


def _batch_clear_widget(widget):
    try:
        widget.value = ()
    except Exception:
        try:
            widget.value = {}
        except Exception:
            pass
    try:
        widget._counter = 0
    except Exception:
        pass


def _batch_save_uploads(value, folder, prefix):
    saved = []
    timestamp = int(time.time() * 1000)
    for index, item in enumerate(_batch_upload_items(value)):
        name = _batch_safe_name(item.get("name", f"image_{index}.png"))
        content = _batch_bytes(item.get("content", b""))
        if not content:
            continue
        path = folder / f"{prefix}_{timestamp}_{index:03d}_{name}"
        with open(path, "wb") as file_obj:
            file_obj.write(content)
        saved.append((str(path), item.get("name", name)))
    return saved


def _batch_read_image(path):
    """Read with EXIF orientation applied, then convert RGB to OpenCV BGR."""
    try:
        with PILImage.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            return cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2BGR)
    except Exception:
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Could not read image: {Path(path).name}")
        return image


def _batch_write_image(path, image):
    path = str(path)
    if path.lower().endswith(".png"):
        ok = cv2.imwrite(path, image, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    else:
        ok = cv2.imwrite(path, image, [cv2.IMWRITE_JPEG_QUALITY, 98])
    if not ok:
        raise IOError(f"Could not save output image: {path}")


def _batch_make_thumbnail(path, key, max_side=420):
    image = _batch_read_image(path)
    height, width = image.shape[:2]
    scale = min(1.0, float(max_side) / max(height, width))
    if scale < 1.0:
        image = cv2.resize(
            image,
            (max(1, int(width * scale)), max(1, int(height * scale))),
            interpolation=cv2.INTER_AREA,
        )
    thumb_path = BATCH_PREVIEW_DIR / f"{key}.jpg"
    cv2.imwrite(str(thumb_path), image, [cv2.IMWRITE_JPEG_QUALITY, 88])
    with open(thumb_path, "rb") as file_obj:
        encoded = base64.b64encode(file_obj.read()).decode("ascii")
    return "data:image/jpeg;base64," + encoded


def _batch_unique_output_path(original_name, extension):
    safe = _batch_safe_name(original_name)
    stem = Path(safe).stem
    candidate = BATCH_OUTPUT_DIR / f"{stem}_swapped{extension}"
    number = 2
    while candidate.exists():
        candidate = BATCH_OUTPUT_DIR / f"{stem}_swapped_{number}{extension}"
        number += 1
    return candidate


def process_one_image_face_swap(
    source_face,
    target_path,
    target_name,
    face_mode="largest",
    use_gfpgan=True,
    gfpgan_weight=0.70,
    output_extension=".png",
):
    """Swap one or all target faces while keeping the original image resolution."""
    target_image = _batch_read_image(target_path)
    target_faces = detect_faces_fast(target_image)
    if not target_faces:
        raise ValueError("No target face detected")

    if face_mode == "all":
        selected_faces = sorted(target_faces, key=_face_area, reverse=True)
    else:
        selected_faces = [select_largest_face(target_faces)]

    result = target_image.copy()
    changed_faces = 0
    for target_face in selected_faces:
        before = result.copy()
        candidate = swap_face_fast(result, source_face, target_face)
        roi_change = face_roi_change_ratio(before, candidate, target_face)
        if roi_change >= 0.001:
            result = candidate
            changed_faces += 1

    if changed_faces == 0:
        raise RuntimeError("The model detected a face but produced no visible face change")

    if use_gfpgan:
        result = enhance_face_fast(result, use_gfpgan=True, weight=gfpgan_weight)

    output_path = _batch_unique_output_path(target_name, output_extension)
    _batch_write_image(output_path, result)
    return {
        "input": str(target_path),
        "output": str(output_path),
        "name": target_name,
        "detected_faces": len(target_faces),
        "changed_faces": changed_faces,
        "width": int(result.shape[1]),
        "height": int(result.shape[0]),
    }


_batch_css = """
<style>
.batch-shell {max-width:1040px;padding:16px;border:1px solid #2b2940;border-radius:10px;background:#171827;color:#f5f3ff;font-family:Inter,Arial,sans-serif}
.batch-title {font-size:20px;font-weight:800;margin-bottom:4px}
.batch-note {color:#aaa4ca;font-size:13px;line-height:1.5;margin-bottom:12px}
.batch-status {padding:10px 12px;border-radius:8px;background:#1e1e2c;border:1px solid #30304a;color:#d8d3ff;margin:8px 0}
.batch-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:10px;margin-top:10px}
.batch-result {background:#11101d;border:1px solid #2b2940;border-radius:8px;padding:8px}
.batch-result img {width:100%;height:180px;object-fit:contain;border-radius:6px;background:#090811}
.batch-caption {font-size:12px;color:#c9c3ef;margin-top:6px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.batch-upload-status {padding:9px 12px;border-radius:8px;background:rgba(19,242,163,.08);border:1px solid rgba(19,242,163,.24);color:#c9ffec;margin:8px 0;font-size:13px}
</style>
"""

batch_header = widgets.HTML(
    _batch_css
    + "<div class='batch-shell'><div class='batch-title'>Batch Image Face Swap</div>"
      "<div class='batch-note'>One source face, many target images. High Quality uses the same neural model and GFPGAN setting as video HQ, then saves lossless PNG files. Use only with permission.</div></div>"
)
batch_upload_controls = widgets.Output(layout=widgets.Layout(max_width="1040px"))
batch_face_mode = widgets.Dropdown(
    options=[("Largest face only", "largest"), ("All detected faces", "all")],
    value="largest",
    description="Replace:",
    layout=widgets.Layout(width="300px"),
)
batch_quality = widgets.Dropdown(
    options=[
        ("High Quality - same as video HQ", "HIGH_QUALITY"),
        ("Natural HQ - no restoration", "NATURAL_HQ"),
        ("Fast Batch", "FAST_BATCH"),
    ],
    value="HIGH_QUALITY",
    description="Quality:",
    layout=widgets.Layout(width="390px"),
)
batch_run_button = widgets.Button(
    description="Swap image batch", icon="magic", button_style="primary",
    layout=widgets.Layout(width="190px", height="42px"),
)
batch_clear_button = widgets.Button(
    description="Clear targets", icon="trash", layout=widgets.Layout(width="140px")
)
batch_status = widgets.HTML("<div class='batch-status'>Upload one source face and at least one target image.</div>")
batch_upload_status = widgets.HTML("<div class='batch-upload-status'>Upload status: waiting for file selection.</div>")
batch_file_upload_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="File: 0%", bar_style="info",
    layout=widgets.Layout(width="520px"), style={"description_width": "90px"},
)
batch_overall_upload_progress = widgets.IntProgress(
    value=0, min=0, max=100, description="Overall: 0%", bar_style="info",
    layout=widgets.Layout(width="520px"), style={"description_width": "90px"},
)
batch_progress = widgets.IntProgress(
    value=0, min=0, max=1, description="Process:", bar_style="info",
    layout=widgets.Layout(width="500px"),
)
batch_gallery = widgets.HTML()
batch_log = widgets.Output()


def _batch_set_status(text):
    batch_status.value = f"<div class='batch-status'>{html.escape(str(text))}</div>"


def _batch_set_upload_status(text, file_percent=None, overall_percent=None):
    batch_upload_status.value = f"<div class='batch-upload-status'>{html.escape(str(text))}</div>"
    if file_percent is not None:
        file_percent = max(0, min(100, int(file_percent)))
        batch_file_upload_progress.value = file_percent
        batch_file_upload_progress.description = f"File: {file_percent}%"
    if overall_percent is not None:
        overall_percent = max(0, min(100, int(overall_percent)))
        batch_overall_upload_progress.value = overall_percent
        batch_overall_upload_progress.description = f"Overall: {overall_percent}%"


def _batch_render_state():
    source = _batch_state.get("source_name") or "not uploaded"
    targets = len(_batch_state.get("target_paths", []))
    _batch_set_status(f"Source: {source} | Target images ready: {targets}")


def _batch_render_gallery(results):
    if not results:
        batch_gallery.value = ""
        return
    cards = []
    for index, result in enumerate(results[:12]):
        url = _batch_make_thumbnail(result["output"], f"result_{int(time.time())}_{index}")
        caption = html.escape(Path(result["output"]).name)
        detail = f"{result['width']}x{result['height']} | faces changed: {result['changed_faces']}"
        cards.append(
            f"<div class='batch-result'><img src='{url}'/>"
            f"<div class='batch-caption'>{caption}</div>"
            f"<div class='batch-caption'>{html.escape(detail)}</div></div>"
        )
    suffix = "" if len(results) <= 12 else f"<div class='batch-note'>Showing 12 of {len(results)} results. All files are in the ZIP.</div>"
    batch_gallery.value = _batch_css + "<div class='batch-grid'>" + "".join(cards) + "</div>" + suffix


def _batch_upload_html():
    """Browser-to-Colab chunk uploader with per-file and overall progress."""
    return r"""
    <style>
      .bi-upload-shell {max-width:1040px;padding:14px;border:1px solid #30304a;border-radius:9px;background:#11101d;color:#f5f3ff;font-family:Inter,Arial,sans-serif}
      .bi-upload-row {display:flex;gap:10px;flex-wrap:wrap;align-items:center}
      .bi-upload-btn {border:0;border-radius:7px;background:#6c18ff;color:white;font-weight:800;padding:11px 16px;cursor:pointer}
      .bi-upload-btn.target {background:#4237a8}
      .bi-meter {height:12px;background:#292740;border-radius:999px;overflow:hidden;margin-top:10px}
      .bi-meter > div {height:100%;width:0%;background:linear-gradient(90deg,#6c18ff,#13f2a3);transition:width .12s linear}
      .bi-meter-label {font-size:12px;color:#c9c3ef;margin-top:4px}
      .bi-browser-status {font-size:13px;color:#d8d3ff;margin-top:9px;min-height:18px}
    </style>
    <div class="bi-upload-shell">
      <div class="bi-upload-row">
        <button class="bi-upload-btn" id="bi-source-button">Upload source face</button>
        <button class="bi-upload-btn target" id="bi-target-button">Add target images</button>
        <span style="font-size:12px;color:#aaa4ca">Target selection supports one or many images and appends to the current list.</span>
      </div>
      <input id="bi-source-input" type="file" accept="image/*" style="display:none">
      <input id="bi-target-input" type="file" accept="image/*" multiple style="display:none">
      <div class="bi-meter"><div id="bi-file-bar"></div></div>
      <div class="bi-meter-label" id="bi-file-label">Current file: 0%</div>
      <div class="bi-meter"><div id="bi-overall-bar"></div></div>
      <div class="bi-meter-label" id="bi-overall-label">Overall selection: 0%</div>
      <div class="bi-browser-status" id="bi-browser-status">Waiting for file selection.</div>
    </div>
    <script>
    (() => {
      const sourceButton = document.getElementById('bi-source-button');
      const targetButton = document.getElementById('bi-target-button');
      const sourceInput = document.getElementById('bi-source-input');
      const targetInput = document.getElementById('bi-target-input');
      const fileBar = document.getElementById('bi-file-bar');
      const overallBar = document.getElementById('bi-overall-bar');
      const fileLabel = document.getElementById('bi-file-label');
      const overallLabel = document.getElementById('bi-overall-label');
      const browserStatus = document.getElementById('bi-browser-status');
      const chunkSize = 2 * 1024 * 1024;

      const setMeters = (filePct, overallPct, fileText, overallText) => {
        fileBar.style.width = filePct + '%';
        overallBar.style.width = overallPct + '%';
        fileLabel.textContent = fileText || ('Current file: ' + filePct + '%');
        overallLabel.textContent = overallText || ('Overall selection: ' + overallPct + '%');
      };
      const toBase64 = (buffer) => {
        const bytes = new Uint8Array(buffer);
        let binary = '';
        const step = 0x8000;
        for (let i = 0; i < bytes.length; i += step) {
          binary += String.fromCharCode(...bytes.subarray(i, Math.min(i + step, bytes.length)));
        }
        return btoa(binary);
      };
      const uploadSelection = async (kind, fileList) => {
        const selected = Array.from(fileList || []);
        if (!selected.length) return;
        const totalBytes = selected.reduce((sum, file) => sum + file.size, 0);
        let overallSent = 0;
        sourceButton.disabled = true;
        targetButton.disabled = true;
        setMeters(0, 0);
        try {
          for (let fileIndex = 0; fileIndex < selected.length; fileIndex++) {
            const file = selected[fileIndex];
            const totalChunks = Math.max(1, Math.ceil(file.size / chunkSize));
            const uploadId = kind + '-' + Date.now() + '-' + fileIndex + '-' + Math.random().toString(16).slice(2);
            browserStatus.textContent = 'Uploading ' + (fileIndex + 1) + '/' + selected.length + ': ' + file.name;
            for (let chunkIndex = 0; chunkIndex < totalChunks; chunkIndex++) {
              const start = chunkIndex * chunkSize;
              const end = Math.min(file.size, start + chunkSize);
              const chunk = await file.slice(start, end).arrayBuffer();
              overallSent += chunk.byteLength;
              const filePct = Math.round((end / Math.max(file.size, 1)) * 100);
              const overallPct = Math.round((overallSent / Math.max(totalBytes, 1)) * 100);
              setMeters(
                filePct,
                overallPct,
                'Current file ' + (fileIndex + 1) + '/' + selected.length + ': ' + filePct + '% | ' + file.name,
                'Overall selection: ' + overallPct + '%'
              );
              await google.colab.kernel.invokeFunction(
                'batch_face_swap_upload.receive_chunk',
                [kind, uploadId, file.name, file.type || '', toBase64(chunk), chunkIndex,
                 totalChunks, fileIndex, selected.length, file.size, totalBytes, overallSent],
                {}
              );
            }
          }
          browserStatus.textContent = 'Upload complete: ' + selected.length + ' file(s), 100%.';
          setMeters(100, 100);
        } catch (error) {
          browserStatus.textContent = 'UPLOAD ERROR: ' + error.message;
        } finally {
          sourceButton.disabled = false;
          targetButton.disabled = false;
          sourceInput.value = '';
          targetInput.value = '';
        }
      };
      sourceButton.onclick = () => sourceInput.click();
      targetButton.onclick = () => targetInput.click();
      sourceInput.onchange = event => uploadSelection('source', event.target.files);
      targetInput.onchange = event => uploadSelection('target', event.target.files);
    })();
    </script>
    """


def _batch_refresh_upload_controls():
    with batch_upload_controls:
        clear_output(wait=True)
        display(HTML(_batch_upload_html()))


_batch_upload_sessions = {}


def _batch_receive_upload_chunk(
    kind, upload_id, name, mime, chunk_b64, chunk_index, total_chunks,
    file_index, total_files, total_bytes, total_batch_bytes, overall_sent,
):
    kind = "source" if kind == "source" else "target"
    chunk_index = int(chunk_index)
    total_chunks = max(1, int(total_chunks))
    file_index = int(file_index)
    total_files = max(1, int(total_files))
    total_bytes = int(total_bytes)
    total_batch_bytes = max(1, int(total_batch_bytes))
    overall_sent = int(overall_sent)

    if chunk_index == 0:
        if kind == "source":
            shutil.rmtree(BATCH_SOURCE_DIR, ignore_errors=True)
            BATCH_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
            folder, prefix = BATCH_SOURCE_DIR, "source"
        else:
            BATCH_TARGET_DIR.mkdir(parents=True, exist_ok=True)
            folder, prefix = BATCH_TARGET_DIR, "target"
        safe_name = _batch_safe_name(name)
        path = folder / f"{prefix}_{int(time.time() * 1000)}_{file_index:03d}_{safe_name}"
        with open(path, "wb"):
            pass
        _batch_upload_sessions[upload_id] = {
            "path": str(path), "name": name, "written": 0, "expected": total_bytes,
        }

    session = _batch_upload_sessions.get(upload_id)
    if not session:
        raise RuntimeError("Upload session expired. Select the image(s) again.")
    chunk = base64.b64decode(chunk_b64)
    with open(session["path"], "ab") as file_obj:
        file_obj.write(chunk)
    session["written"] += len(chunk)

    file_percent = int(100 * session["written"] / max(total_bytes, 1))
    overall_percent = int(100 * overall_sent / total_batch_bytes)
    _batch_set_upload_status(
        f"Uploading file {file_index + 1}/{total_files}: {name} | "
        f"file {file_percent}% | overall {overall_percent}%",
        file_percent,
        overall_percent,
    )

    if chunk_index + 1 >= total_chunks:
        saved_size = os.path.getsize(session["path"])
        if saved_size != total_bytes:
            raise IOError(f"Upload size mismatch for {name}: expected {total_bytes}, got {saved_size}")
        if kind == "source":
            _batch_state["source_path"] = session["path"]
            _batch_state["source_name"] = name
        else:
            _batch_state["target_paths"].append(session["path"])
            _batch_state["target_names"].append(name)
        _batch_upload_sessions.pop(upload_id, None)
        _batch_render_state()
        if file_index + 1 >= total_files:
            _batch_set_upload_status(
                f"Upload complete: {total_files} file(s) | 100%",
                100,
                100,
            )
    return {"ok": True, "file_percent": file_percent, "overall_percent": overall_percent}


output.register_callback("batch_face_swap_upload.receive_chunk", _batch_receive_upload_chunk)


def _batch_clear_targets(_):
    shutil.rmtree(BATCH_TARGET_DIR, ignore_errors=True)
    BATCH_TARGET_DIR.mkdir(parents=True, exist_ok=True)
    _batch_state["target_paths"] = []
    _batch_state["target_names"] = []
    _batch_state["results"] = []
    batch_gallery.value = ""
    _batch_set_upload_status("Target list cleared. Add one or many target images.", 0, 0)
    _batch_render_state()


def _batch_build_zip(results):
    zip_path = BATCH_ROOT / "batch_face_swap_results.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for result in results:
            archive.write(result["output"], arcname=Path(result["output"]).name)
    return str(zip_path)


def _batch_run(_):
    batch_run_button.disabled = True
    batch_clear_button.disabled = True
    with batch_log:
        clear_output()
        try:
            source_path = _batch_state.get("source_path")
            target_paths = list(_batch_state.get("target_paths", []))
            target_names = list(_batch_state.get("target_names", []))
            if not source_path:
                raise ValueError("Upload a source face image first")
            if not target_paths:
                raise ValueError("Add at least one target image")

            profile = _BATCH_PROFILES[batch_quality.value]
            try:
                face_app.prepare(
                    ctx_id=0 if torch.cuda.is_available() else -1,
                    det_size=profile["det_size"],
                )
            except Exception as prepare_error:
                print(f"Detector already prepared or re-prepare skipped: {prepare_error}")

            source_image = _batch_read_image(source_path)
            source_faces = detect_faces_fast(source_image)
            if not source_faces:
                raise ValueError("No face detected in the source image. Try a clear, front-facing photo.")
            source_face = select_largest_face(source_faces)

            shutil.rmtree(BATCH_OUTPUT_DIR, ignore_errors=True)
            BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            batch_progress.max = len(target_paths)
            batch_progress.value = 0
            batch_progress.bar_style = "info"
            results = []
            failures = []
            started = time.perf_counter()

            for index, (target_path, target_name) in enumerate(zip(target_paths, target_names), 1):
                _batch_set_status(f"Processing {index}/{len(target_paths)}: {target_name}")
                try:
                    result = process_one_image_face_swap(
                        source_face=source_face,
                        target_path=target_path,
                        target_name=target_name,
                        face_mode=batch_face_mode.value,
                        use_gfpgan=profile["gfpgan"],
                        gfpgan_weight=profile["gfpgan_weight"],
                        output_extension=profile["extension"],
                    )
                    results.append(result)
                    print(
                        f"OK {target_name}: {result['changed_faces']}/"
                        f"{result['detected_faces']} face(s), {result['width']}x{result['height']}"
                    )
                except Exception as image_error:
                    failures.append((target_name, str(image_error)))
                    print(f"SKIPPED {target_name}: {image_error}")
                batch_progress.value = index

            if not results:
                raise RuntimeError("No images were successfully processed. See the messages above.")

            _batch_state["results"] = results
            _batch_render_gallery(results)
            elapsed = time.perf_counter() - started
            batch_progress.bar_style = "success"
            _batch_set_status(
                f"Done: {len(results)} image(s) saved, {len(failures)} skipped, "
                f"{elapsed:.1f} seconds | Lossless PNG output"
            )

            if len(results) == 1:
                download_path = results[0]["output"]
            else:
                download_path = _batch_build_zip(results)
            print("=" * 70)
            print(f"Completed: {len(results)} | Skipped: {len(failures)}")
            print(f"Download: {download_path}")
            files.download(download_path)
        except Exception as error:
            batch_progress.bar_style = "danger"
            _batch_set_status(f"ERROR: {error}")
            print(f"ERROR: {error}")
        finally:
            batch_run_button.disabled = False
            batch_clear_button.disabled = False


batch_clear_button.on_click(_batch_clear_targets)
batch_run_button.on_click(_batch_run)
_batch_refresh_upload_controls()

display(
    widgets.VBox(
        [
            batch_header,
            batch_upload_controls,
            batch_upload_status,
            batch_file_upload_progress,
            batch_overall_upload_progress,
            widgets.HBox([batch_clear_button]),
            widgets.HBox([batch_quality, batch_face_mode, batch_run_button]),
            batch_status,
            batch_progress,
            batch_gallery,
            batch_log,
        ],
        layout=widgets.Layout(max_width="1040px", gap="10px"),
    )
)
